<!-- your logo here! -->
<img src="https://github.com/bionadmin/distrib/raw/main/core.png"/>

# Attempt to update the Silver schema without losing data.

#### Debug options

In [1]:
debug_mode = False # Include debug select and print statements
force_remount = False #Normally mount points are left alone if they exists. Setting this to true will replace the mountpoints.
force_rebuild_of_all_tables = False

StatementMeta(, 38e9d83f-b5c6-4cd8-8530-47047403425f, 3, Finished, Available, Finished, False)

#### Some items we need...

In [2]:
from datetime import datetime
now = datetime.now()
tag = now.strftime("%Y%m%d%H%M%S")
print(tag)

useManagedTables = True
unmanagedArchiveCopies = 1
managedTablesKeepOldTableVersions = False
rebuildTableWhenColumnOrderChanges = True

destination_database_name="LH_BI_SILVER_TIER1"
spark.conf.set("sess.NB_BRNZ_TO_SIL_PLEX_Default_Table_destination_database_name", destination_database_name)



dest_base_path= "<Dest Folder>"

dest_additional_path="Silver/"
dest_base_path=dest_base_path.replace("<Dest Folder>",dest_additional_path)
destination_file_folder=""#set to None if no subfolder is desired. Include only internal slashes in the path.
if destination_file_folder==None or destination_file_folder=="":
    destination_database_default_path= dest_base_path
else:
    destination_database_default_path= dest_base_path + "/"+destination_file_folder
destination_filepath=destination_database_default_path+"/"

spark.conf.set("sess.NB_BRNZ_TO_SIL_PLEX_Default_Table_destination_database_default_path", destination_database_default_path)

StatementMeta(, 38e9d83f-b5c6-4cd8-8530-47047403425f, 4, Finished, Available, Finished, False)

20260226210645


#### Misc. Spark Options
If generating code, edit these in the template settings for consistency<br>
across all notebooks.

In [3]:
spark.conf.set("spark.sql.parquet.vorder.enabled","false")
spark.conf.set("spark.microsoft.delta.optimizeWrite.enabled","true")
spark.conf.set("spark.microsoft.delta.optimizeWrite.binSize","1073741824")

StatementMeta(, 38e9d83f-b5c6-4cd8-8530-47047403425f, 5, Finished, Available, Finished, False)

In [4]:
%%sql
SET ANSI_MODE = TRUE

StatementMeta(, 38e9d83f-b5c6-4cd8-8530-47047403425f, 6, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 2 fields>

In [5]:
def TableArraysAreEqual(old, new):
    if (len(old)!=len(new)):
        return False
    outerIndex=0
    for rowNew in new:
        innerIndex=0
        found = False
        for rowOld in old:
            if rowNew["col_name"]==rowOld["col_name"] and rowNew["data_type"]==rowOld["data_type"] and (rowNew["comment"] or "")==(rowOld["comment"] or ""):
                found=True
                break
            innerIndex=innerIndex+1
        if not found:
            if debug_mode:
                    print("Table structure has changed. Rebuilding table.")
            return False
        if rebuildTableWhenColumnOrderChanges and innerIndex!=outerIndex:
            if debug_mode:
                    print("Column Order has changed. Rebuilding table.")
            return False  
        outerIndex=outerIndex+1
    return True

def GetColumnByComment(arr,comment):
    if comment == "" or comment is None:
        return None
    for row in arr:
        if (row["comment"] is not None):
            if (comment in row["comment"]):
                return row
    return None

def GetColumnByName(arr, name):
    for row in arr:
        if (name == row["col_name"]):
            return row
    return None

def GetColumnByCommentThenName(arr, comment, name):
    row = GetColumnByComment(arr,comment)
    if row is None:
        row = GetColumnByName(arr,name)
    return row

def GenerateTableCreate(new, databaseName, tableName, partitions,location):

    stmt = "CREATE OR REPLACE TABLE "+databaseName+"."+tableName
    stmt=stmt+"\r\n("
    for row in new:
        stmt = stmt+"\r\n\t`"+row["col_name"]+"` "+row["data_type"]
        if row["comment"]:
            stmt = stmt +" COMMENT \""+row["comment"]+"\","
        else:
            stmt = stmt +","
    stmt=stmt[0:len(stmt)-1]#get rid of last trailing comma
    stmt=stmt+"\r\n)"  
    stmt=stmt+"\r\nUSING DELTA"
    if partitions is not None:
        if len(partitions)>0:
            stmt=stmt+"\r\nPARTITIONED BY("
            firstPartition=True
            for par in partitions:
                if not firstPartition:
                    stmt+=","
                firstPartition=False
                stmt+="`"+par+"`"
            stmt=stmt+")"
    if not useManagedTables:
        stmt=stmt+"\r\nLOCATION \""+location+"\""
    if debug_mode:
        print (stmt)
    return stmt

def GenerateTableCopy(old, new, databaseName, oldTableName, newTableName):
    stmt = "INSERT INTO "+databaseName+"."+newTableName
    stmt=stmt+"\r\n("
    for row in new:
        stmt = stmt+"\r\n\t`"+row["col_name"]+"`,"
    stmt=stmt[0:len(stmt)-1]#get rid of last trailing comma
    stmt=stmt+"\r\n)"   
    stmt=stmt+"SELECT "
    for row in new:
        oldRow = GetColumnByCommentThenName(old,row["comment"],row["col_name"])
        if oldRow is None:
            stmt=stmt+"\r\n\tNULL AS `"+row["col_name"]+"`,"
        else:
            stmt=stmt+"\r\n\tCAST(`"+oldRow["col_name"]+"` AS "+row["data_type"]+") AS `"+row["col_name"]+"`,"
    stmt=stmt[0:len(stmt)-1]#get rid of last trailing comma
    stmt=stmt+"\r\nFROM "+databaseName+"."+oldTableName
    if debug_mode:
        print (stmt)
    return stmt
 
def GenerateTableRename(databaseName, oldTableName, newTableName):
    stmt = "ALTER TABLE "+databaseName+"."+oldTableName +" RENAME TO "+newTableName
    if debug_mode:
        print (stmt)
    return stmt



def RemoveNonColumns(arr):
    returnArr = []
    for row in arr:
        if row["col_name"] == "# Partitioning":#Old way
            break  # we've gone past the columns. all done
        if row["col_name"] == "# Partition Information":#New Way
            break  # we've gone past the columns. all done
        if row["col_name"] == "# col_name":#New Way
            break  # we've gone past the columns. all done
        if row["data_type"] != "" and row["data_type"] is not None:
            returnArr.append(row)
    return returnArr
  
def TableExists(databaseName, tableName):
    table_list=spark.sql("show tables in "+databaseName)
    table_name=table_list.filter(table_list.tableName==tableName.lower()).collect()
    if table_name:
        return True
    else:
        return False
    
def UnmanagedRenameFolder(oldlocation,newlocation):
    if useManagedTables:
        return
    notebookutils.fs.mv(oldlocation, newlocation, True)

def UnmanagedDropTable(databaseName, tableName):
    spark.sql("DROP TABLE "+databaseName+"."+tableName)

def ManagedDropTable(databaseName, tableName):
    spark.sql("DROP TABLE "+databaseName+"."+tableName)
    
def UpdateTableStructure(old, new, databaseName, tableName, partitions, location, force):
    
    if (not force) and TableArraysAreEqual(old, new):
        if debug_mode:
            print ("Tables are the same. No need to update.")
        return
    stmt = GenerateTableCreate(new,databaseName, tableName+"_"+tag+"_TMP", partitions,location+"_"+tag+"_TMP")
    spark.sql(stmt)
    stmt = GenerateTableCopy(old,new,databaseName, tableName,tableName+"_"+tag+"_TMP")
    spark.sql(stmt)
    if useManagedTables:
        #With managed tables, the folder gets renamed with the table so this is all we need. Nice and simple.
        stmt = GenerateTableRename(databaseName,tableName, tableName+"_"+tag)
        spark.sql(stmt)
        stmt = GenerateTableRename(databaseName, tableName+"_"+tag+"_TMP",tableName)
        spark.sql(stmt)
        if not managedTablesKeepOldTableVersions:
            ManagedDropTable(databaseName, tableName+"_"+tag)
    if not useManagedTables:
        #with unmanaged, we need to rename the folder too so we don't have the new table's data sitting in a folder with a weird name...
        #basically since these are unmanaged we move the data around and then recreate the table which just updates the meta-data
        UnmanagedDropTable(databaseName,tableName)#drop our two tables.
        UnmanagedDropTable(databaseName,tableName+"_"+tag+"_TMP")#drop our two tables.
        UnmanagedRenameFolder(location,location+"_"+tag+"_PreviousVersion")#move the old version of the table's data out to a previous version folder.
        UnmanagedRenameFolder(location+"_"+tag+"_TMP",location)#move the new tables data into the folder with the basic table name
        stmt=GenerateTableCreate(new,databaseName, tableName, partitions,location)#create the table against the new path.
        if debug_mode:
            print(stmt)
        spark.sql(stmt)
        
def GetPartitions(collectedDescription):
    returnArr = []
    columns = RemoveNonColumns(collectedDescription)
    foundPartitionMarker = False
    newWay = False
    for row in collectedDescription:
        if row["col_name"] == "# Partitioning":  #older way
            foundPartitionMarker = True
            continue
        if row["col_name"] == "# Partition Information":  # newer way
            foundPartitionMarker = True
            newWay=True
            continue
        else:
            if not foundPartitionMarker:
                continue
        # if we made it this far, we are on the partitions section of the description.
        if row["col_name"] == "Not partitioned":
            return returnArr
        if newWay:
            potentialName = row["col_name"]
            if not potentialName.startswith("#"):
                for col in columns:
                    # confirm we are looking at a partition column
                    if col["col_name"].lower() == potentialName.lower():
                        returnArr.append(potentialName)
                        break
        else:
            if row["col_name"].startswith("Part "):  # partitions show up this way.
                potentialName = row[
                    "data_type"
                ]  # partition column names are in the type column.
                for col in columns:
                    # confirm we are looking at a partition column
                    if col["col_name"].lower() == potentialName.lower():
                        returnArr.append(potentialName)
                        break
    return returnArr


def StringArraysHaveSameElementsCI(one,two):
    if len(one)!=len(two):
        return False
    for strone in one:
        found = False
        for strtwo in two:
            if strone.lower()==strtwo.lower():
                found=True
                break
        if not found:
            return False
    return True
    
def RemoveOldArchiveCopies(tableName, filePath):
    if not useManagedTables:
        if unmanagedArchiveCopies>-1:#-1 means keep all copies!
            directoriesToParse=[]
            for fileinfo in notebookutils.fs.ls(filePath):
                name = fileinfo.name
                if name.startswith(tableName) and len(name)==len(tableName)+32 and name.endswith("_PreviousVersion/"):
                    if debug_mode:
                        print (fileinfo.path)
                    directoriesToParse.append(fileinfo.path)
            directoriesToParse.sort(reverse=True)
            filesToKeep = unmanagedArchiveCopies
            for directory in directoriesToParse:
                if filesToKeep>0:
                    filesToKeep=filesToKeep-1
                    continue
                else:
                    if debug_mode:
                        print("Removing archive directory "+directory)
                    notebookutils.fs.rm(directory,True)
                    
def TableIsManaged(databaseName, tableName):
    df = spark.sql("DESCRIBE EXTENDED "+databaseName+"."+tableName)
    arr = df.collect()
    foundDetailed = False
    for row in arr:
        if row["col_name"]=="# Detailed Table Information":
            foundDetailed=True
            continue
        if not foundDetailed:
            continue
        if row["col_name"]=="Type":
            if row["data_type"]=="MANAGED":
                return True
    return False

def ProcessTable(databaseName, tableName, comparisonArray, partitions, path):
    existingTableIsManaged=False
    tableExists = TableExists(databaseName,tableName)
    if tableExists:
        if debug_mode:
            print ("Table Exists")
        description = spark.sql("DESCRIBE "+databaseName+"."+tableName)
        oldpartitions = GetPartitions(description.collect())
        force = not StringArraysHaveSameElementsCI(oldpartitions,partitions)#rewrite table if partitions changed
        if force_rebuild_of_all_tables:
            force = True
        existingTableIsManaged = TableIsManaged(databaseName,tableName)
        if not force:  
            if (useManagedTables and not existingTableIsManaged) or (not useManagedTables and existingTableIsManaged):
                force = True
                if debug_mode:
                    if useManagedTables:
                        print("Converting tables from unmanaged to managed...")
                    else:
                        print("Converting tables from managed to unmanaged...")
        cols = RemoveNonColumns(description.collect())
        UpdateTableStructure(cols, comparisonArray, databaseName,tableName,partitions,path,force)
    else:
        if debug_mode:
            print ("Table Does not Exist. Creating Table...")
        stmt = GenerateTableCreate(comparisonArray,databaseName,tableName,partitions,path)
        spark.sql(stmt)

    if not existingTableIsManaged:
        RemoveOldArchiveCopies(tableName,destination_filepath)

StatementMeta(, 38e9d83f-b5c6-4cd8-8530-47047403425f, 7, Finished, Available, Finished, False)

# master for test , use this cell to create table

## Process Plex_query_exploded_bom

In [6]:


# Provide Table-specific columns: (name, data_type, comment)
base_columns = [
('PCN','int',''),
('Version','String',''),
('cost_model_key','int',''),
('Child_Key','int',''),
('Part_Operation_Key','int',''),
('Parent_Part_Key','int',''),
('parent_part_no','string',''),
('Description','string',''),
('Revision','string',''),
('Name','string',''),
('Operation_No','int',''),
('Operation_Code','string',''),
('Cost_Sub_Type_Key','int',''),
('Cost_Type','string',''),
('Cost_Sub_Type','string',''),
('Quantity','decimal(16,6)',''),
('Cost','decimal(16,6)',''),
('Level','int',''),
('Path','string',''),
('CT_Sort_Order','int',''),
('CST_Sort_Order','int',''),
('PartRowSpan','int',''),
('OpRowSpan','int',''),
('Note','string',''),
('Calc_Note','string',''),
('zero_level_desc','string',''),

]

# Audit-prefixed columns to include for every table
audit_columns = [ 
    ('Audit_SourceFileOrFolder',      'string',     ''), 
    ('Audit_CreatedDateTime',         'timestamp',  ''), 
    ('Audit_ModifiedDateTime',        'timestamp',  ''),
    ('Audit_Source_ModifiedDateTime',        'timestamp',  ''),
    ('AuditPipelineTriggerTime',        'timestamp',  ''),
    ('AuditPipelineRunId',        'string',  ''),
    ('AuditFilePath',        'string',  ''),
    ('RevisionHash',        'binary',  ''),
    ]

# Combine base and audit columns
all_columns = base_columns + audit_columns


comparisonArray = [
    {
        "col_name":    col,
        "data_type":   dt,
        "comment":     comment
    }
    for col, dt, comment in all_columns
]

partitions = []

# Table alias and path
table_alias = "plex_query_exploded_bom"
path = destination_filepath + table_alias

# Invoke ProcessTable
ProcessTable(
    "LH_BI_SILVER_TIER1",
    table_alias,
    comparisonArray,
    partitions,
    path
)


StatementMeta(, e00e84cf-a8f6-49bf-a364-9470ed7caf20, 8, Finished, Available, Finished, False)

## Process Operation

In [7]:


# Provide Table-specific columns: (name, data_type, comment)
base_columns = [
('Plexus_Customer_No',          'int',            ''),
('Operation_Key',               'int',            ''),
('Operation_Code',             'string',            ''),
]

# Audit-prefixed columns to include for every table 
audit_columns = [ 
    ('Audit_SourceFileOrFolder',      'string',     ''), 
    ('Audit_CreatedDateTime',         'timestamp',  ''), 
    ('Audit_ModifiedDateTime',        'timestamp',  ''),
    ('Audit_Source_ModifiedDateTime',        'timestamp',  ''),
    ('AuditPipelineTriggerTime',        'timestamp',  ''),
    ('AuditPipelineRunId',        'string',  ''),
    ('AuditFilePath',        'string',  ''),
    ('RevisionHash',        'binary',  ''),
    ]

# Combine base and audit columns
all_columns = base_columns + audit_columns


comparisonArray = [
    {
        "col_name":    col,
        "data_type":   dt,
        "comment":     comment
    }
    for col, dt, comment in all_columns
]

partitions = []

# Table alias and path
table_alias = "operation"
path = destination_filepath + table_alias

# Invoke ProcessTable
ProcessTable(
    "LH_BI_SILVER_TIER1",
    table_alias,
    comparisonArray,
    partitions,
    path
)


StatementMeta(, e00e84cf-a8f6-49bf-a364-9470ed7caf20, 9, Finished, Available, Finished, False)

## Process Item_type

In [8]:


# Provide Table-specific columns: (name, data_type, comment)
base_columns = [
('Plexus_Customer_No',          'int',            ''),
('Item_Type_Key',               'int',            ''),
('Value_Inventory',             'smallint',            ''),
]

# Audit-prefixed columns to include for every table 
audit_columns = [ 
    ('Audit_SourceFileOrFolder',      'string',     ''), 
    ('Audit_CreatedDateTime',         'timestamp',  ''), 
    ('Audit_ModifiedDateTime',        'timestamp',  ''),
    ('Audit_Source_ModifiedDateTime',        'timestamp',  ''),
    ('AuditPipelineTriggerTime',        'timestamp',  ''),
    ('AuditPipelineRunId',        'string',  ''),
    ('AuditFilePath',        'string',  ''),
    ('RevisionHash',        'binary',  ''),
    ]

# Combine base and audit columns
all_columns = base_columns + audit_columns


comparisonArray = [
    {
        "col_name":    col,
        "data_type":   dt,
        "comment":     comment
    }
    for col, dt, comment in all_columns
]

partitions = []

# Table alias and path
table_alias = "item_type"
path = destination_filepath + table_alias

# Invoke ProcessTable
ProcessTable(
    "LH_BI_SILVER_TIER1",
    table_alias,
    comparisonArray,
    partitions,
    path
)


StatementMeta(, e00e84cf-a8f6-49bf-a364-9470ed7caf20, 10, Finished, Available, Finished, False)

## Process item

In [9]:


# Provide Table-specific columns: (name, data_type, comment)
base_columns = [
('Plexus_Customer_No',          'int',            ''),
('Item_Key',                    'int',            ''),
('Item_No',                     'string',   ''),
('Revision',                    'string',    ''),
('Description',                 'string',   ''),
('Item_Type_Key',               'int',            ''),
]

# Audit-prefixed columns to include for every table 
audit_columns = [ 
    ('Audit_SourceFileOrFolder',      'string',     ''), 
    ('Audit_CreatedDateTime',         'timestamp',  ''), 
    ('Audit_ModifiedDateTime',        'timestamp',  ''),
    ('Audit_Source_ModifiedDateTime',        'timestamp',  ''),
    ('AuditPipelineTriggerTime',        'timestamp',  ''),
    ('AuditPipelineRunId',        'string',  ''),
    ('AuditFilePath',        'string',  ''),
    ('RevisionHash',        'binary',  ''),
    ]

# Combine base and audit columns
all_columns = base_columns + audit_columns


comparisonArray = [
    {
        "col_name":    col,
        "data_type":   dt,
        "comment":     comment
    }
    for col, dt, comment in all_columns
]

partitions = []

# Table alias and path
table_alias = "item"
path = destination_filepath + table_alias

# Invoke ProcessTable
ProcessTable(
    "LH_BI_SILVER_TIER1",
    table_alias,
    comparisonArray,
    partitions,
    path
)


StatementMeta(, e00e84cf-a8f6-49bf-a364-9470ed7caf20, 11, Finished, Available, Finished, False)

## Process Cost_BOM

In [10]:


# Provide Table-specific columns: (name, data_type, comment)
base_columns = [
('PCN',                         'int',            ''),
('Cost_BOM_Key',                'int',            ''),
('Part_Key',                    'int',            ''),
('Material_Key',                 'int',            ''),
('Cost_Model_Key',              'int',            ''),
('Part_Operation_Key',          'int',            ''),
]

# Audit-prefixed columns to include for every table 
audit_columns = [ 
    ('Audit_SourceFileOrFolder',      'string',     ''), 
    ('Audit_CreatedDateTime',         'timestamp',  ''), 
    ('Audit_ModifiedDateTime',        'timestamp',  ''),
    ('Audit_Source_ModifiedDateTime',        'timestamp',  ''),
    ('AuditPipelineTriggerTime',        'timestamp',  ''),
    ('AuditPipelineRunId',        'string',  ''),
    ('AuditFilePath',        'string',  ''),
    ('RevisionHash',        'binary',  ''),
    ]

# Combine base and audit columns
all_columns = base_columns + audit_columns


comparisonArray = [
    {
        "col_name":    col,
        "data_type":   dt,
        "comment":     comment
    }
    for col, dt, comment in all_columns
]

partitions = []

# Table alias and path
table_alias = "cost_BOM"
path = destination_filepath + table_alias

# Invoke ProcessTable
ProcessTable(
    "LH_BI_SILVER_TIER1",
    table_alias,
    comparisonArray,
    partitions,
    path
)


StatementMeta(, e00e84cf-a8f6-49bf-a364-9470ed7caf20, 12, Finished, Available, Finished, False)

## Process Approved_supplier

In [11]:


# Provide Table-specific columns: (name, data_type, comment)
base_columns = [
('Plexus_Customer_No',          'int',            ''),
('Part_Key',                    'int',            ''),
('Part_Operation_Key',          'int',            ''),
('Supplier_No',                 'int',            ''),
('Sort_Order',                  'int',            ''),
]

# Audit-prefixed columns to include for every table 
audit_columns = [ 
    ('Audit_SourceFileOrFolder',      'string',     ''), 
    ('Audit_CreatedDateTime',         'timestamp',  ''), 
    ('Audit_ModifiedDateTime',        'timestamp',  ''),
    ('Audit_Source_ModifiedDateTime',        'timestamp',  ''),
    ('AuditPipelineTriggerTime',        'timestamp',  ''),
    ('AuditPipelineRunId',        'string',  ''),
    ('AuditFilePath',        'string',  ''),
    ('RevisionHash',        'binary',  ''),
    ]

# Combine base and audit columns
all_columns = base_columns + audit_columns


comparisonArray = [
    {
        "col_name":    col,
        "data_type":   dt,
        "comment":     comment
    }
    for col, dt, comment in all_columns
]

partitions = []

# Table alias and path
table_alias = "approved_supplier"
path = destination_filepath + table_alias

# Invoke ProcessTable
ProcessTable(
    "LH_BI_SILVER_TIER1",
    table_alias,
    comparisonArray,
    partitions,
    path
)


StatementMeta(, e00e84cf-a8f6-49bf-a364-9470ed7caf20, 13, Finished, Available, Finished, False)

## Process material_status

In [12]:


# Provide Table-specific columns: (name, data_type, comment)
base_columns = [
('PCN',                         'int',            ''),
('Material_Status_Key',         'int',            ''),
('Active',                      'smallint',            ''),
]

# Audit-prefixed columns to include for every table 
audit_columns = [ 
    ('Audit_SourceFileOrFolder',      'string',     ''), 
    ('Audit_CreatedDateTime',         'timestamp',  ''), 
    ('Audit_ModifiedDateTime',        'timestamp',  ''),
    ('Audit_Source_ModifiedDateTime',        'timestamp',  ''),
    ('AuditPipelineTriggerTime',        'timestamp',  ''),
    ('AuditPipelineRunId',        'string',  ''),
    ('AuditFilePath',        'string',  ''),
    ('RevisionHash',        'binary',  ''),
    ]

# Combine base and audit columns
all_columns = base_columns + audit_columns


comparisonArray = [
    {
        "col_name":    col,
        "data_type":   dt,
        "comment":     comment
    }
    for col, dt, comment in all_columns
]

partitions = []

# Table alias and path
table_alias = "material_status"
path = destination_filepath + table_alias

# Invoke ProcessTable
ProcessTable(
    "LH_BI_SILVER_TIER1",
    table_alias,
    comparisonArray,
    partitions,
    path
)


StatementMeta(, e00e84cf-a8f6-49bf-a364-9470ed7caf20, 14, Finished, Available, Finished, False)

## Process part_MATERIAL

In [13]:


# Provide Table-specific columns: (name, data_type, comment)
base_columns = [
('Plexus_Customer_No',          'int',            ''),
('Part_Material_Key',           'int',            ''),
('Part_Key',                    'int',            ''),
('Material_Key',                'int',            ''),
('Active',                      'smallint',            ''),
('Gross_Weight',                'decimal(16,6)',  ''),
('Sort_Order',                  'decimal(16,6)',            ''),
]

# Audit-prefixed columns to include for every table 
audit_columns = [ 
    ('Audit_SourceFileOrFolder',      'string',     ''), 
    ('Audit_CreatedDateTime',         'timestamp',  ''), 
    ('Audit_ModifiedDateTime',        'timestamp',  ''),
    ('Audit_Source_ModifiedDateTime',        'timestamp',  ''),
    ('AuditPipelineTriggerTime',        'timestamp',  ''),
    ('AuditPipelineRunId',        'string',  ''),
    ('AuditFilePath',        'string',  ''),
    ('RevisionHash',        'binary',  ''),
    ]

# Combine base and audit columns
all_columns = base_columns + audit_columns


comparisonArray = [
    {
        "col_name":    col,
        "data_type":   dt,
        "comment":     comment
    }
    for col, dt, comment in all_columns
]

partitions = []

# Table alias and path
table_alias = "part_material"
path = destination_filepath + table_alias

# Invoke ProcessTable
ProcessTable(
    "LH_BI_SILVER_TIER1",
    table_alias,
    comparisonArray,
    partitions,
    path
)


StatementMeta(, e00e84cf-a8f6-49bf-a364-9470ed7caf20, 15, Finished, Available, Finished, False)

## Process material

In [14]:


# Provide Table-specific columns: (name, data_type, comment)
base_columns = [
('Plexus_Customer_No',          'int',            ''),
('Material_Key',                'int',            ''),
('Material_Code',               'string',   ''),
('Revision',                    'string',    ''),
('Grade',                       'string',   ''),
('Material_Status_Key',         'int',            ''),
]

# Audit-prefixed columns to include for every table 
audit_columns = [ 
    ('Audit_SourceFileOrFolder',      'string',     ''), 
    ('Audit_CreatedDateTime',         'timestamp',  ''), 
    ('Audit_ModifiedDateTime',        'timestamp',  ''),
    ('Audit_Source_ModifiedDateTime',        'timestamp',  ''),
    ('AuditPipelineTriggerTime',        'timestamp',  ''),
    ('AuditPipelineRunId',        'string',  ''),
    ('AuditFilePath',        'string',  ''),
    ('RevisionHash',        'binary',  ''),
    ]

# Combine base and audit columns
all_columns = base_columns + audit_columns


comparisonArray = [
    {
        "col_name":    col,
        "data_type":   dt,
        "comment":     comment
    }
    for col, dt, comment in all_columns
]

partitions = []

# Table alias and path
table_alias = "material"
path = destination_filepath + table_alias

# Invoke ProcessTable
ProcessTable(
    "LH_BI_SILVER_TIER1",
    table_alias,
    comparisonArray,
    partitions,
    path
)


StatementMeta(, e00e84cf-a8f6-49bf-a364-9470ed7caf20, 16, Finished, Available, Finished, False)

## Process Cost_component

In [15]:


# Provide Table-specific columns: (name, data_type, comment)
base_columns = [
('PCN',                         'int',            ''),
('Cost_Model_Key',              'int',            ''),
('Cost_Component_Key',          'bigint',            ''),
('Part_Key',                    'int',            ''),
('Part_Operation_Key',          'int',            ''),
('Cost_Sub_Type_Key',           'int',            ''),
('Cost',                        'decimal(10,6)',  ''),
('Material_Key',                'int',            ''),
('Item_Key',                    'int',            ''),
('Note',                        'string',  ''),
('Calc_Note',                   'string',  ''),
]

# Audit-prefixed columns to include for every table 
audit_columns = [ 
    ('Audit_SourceFileOrFolder',      'string',     ''), 
    ('Audit_CreatedDateTime',         'timestamp',  ''), 
    ('Audit_ModifiedDateTime',        'timestamp',  ''),
    ('Audit_Source_ModifiedDateTime',        'timestamp',  ''),
    ('AuditPipelineTriggerTime',        'timestamp',  ''),
    ('AuditPipelineRunId',        'string',  ''),
    ('AuditFilePath',        'string',  ''),
    ('RevisionHash',        'binary',  ''),
    ]

# Combine base and audit columns
all_columns = base_columns + audit_columns


comparisonArray = [
    {
        "col_name":    col,
        "data_type":   dt,
        "comment":     comment
    }
    for col, dt, comment in all_columns
]

partitions = []

# Table alias and path
table_alias = "cost_component"
path = destination_filepath + table_alias

# Invoke ProcessTable
ProcessTable(
    "LH_BI_SILVER_TIER1",
    table_alias,
    comparisonArray,
    partitions,
    path
)


StatementMeta(, e00e84cf-a8f6-49bf-a364-9470ed7caf20, 17, Finished, Available, Finished, False)

## Process Multi_Out

In [16]:


# Provide Table-specific columns: (name, data_type, comment)
base_columns = [
('PCN',                         'int',            ''),
('Multi_Out_Key',				'int',			  ''),
('Out_Part_Key',                'int',            ''),
('Out_Part_Operation_Key',      'int',            ''),
('Part_Key',                    'int',            ''),
('Part_Operation_Key',          'int',            ''),
('Primary',                     'boolean',            ''),
]

# Audit-prefixed columns to include for every table 
audit_columns = [ 
    ('Audit_SourceFileOrFolder',      'string',     ''), 
    ('Audit_CreatedDateTime',         'timestamp',  ''), 
    ('Audit_ModifiedDateTime',        'timestamp',  ''),
    ('Audit_Source_ModifiedDateTime',        'timestamp',  ''),
    ('AuditPipelineTriggerTime',        'timestamp',  ''),
    ('AuditPipelineRunId',        'string',  ''),
    ('AuditFilePath',        'string',  ''),
    ('RevisionHash',        'binary',  ''),
    ]

# Combine base and audit columns
all_columns = base_columns + audit_columns


comparisonArray = [
    {
        "col_name":    col,
        "data_type":   dt,
        "comment":     comment
    }
    for col, dt, comment in all_columns
]

partitions = []

# Table alias and path
table_alias = "multi_out"
path = destination_filepath + table_alias

# Invoke ProcessTable
ProcessTable(
    "LH_BI_SILVER_TIER1",
    table_alias,
    comparisonArray,
    partitions,
    path
)


StatementMeta(, e00e84cf-a8f6-49bf-a364-9470ed7caf20, 18, Finished, Available, Finished, False)

## Process part_bom

In [17]:


# Provide Table-specific columns: (name, data_type, comment)
base_columns = [
('Plexus_Customer_No',          'int',            ''),
('BOM_Key',                     'int',            ''),
('Part_Key',                    'int',            ''),
('Component_Part_Key',          'int',            ''),
('Part_Operation_Key',          'int',            ''),
('Active',                      'smallint',            ''),
('Side_Key',                    'int',            ''),
('Quantity',                    'decimal(16,6)',  ''),
('Purchasing_Item_Key',         'int',            ''),
]

# Audit-prefixed columns to include for every table 
audit_columns = [ 
    ('Audit_SourceFileOrFolder',      'string',     ''), 
    ('Audit_CreatedDateTime',         'timestamp',  ''), 
    ('Audit_ModifiedDateTime',        'timestamp',  ''),
    ('Audit_Source_ModifiedDateTime',        'timestamp',  ''),
    ('AuditPipelineTriggerTime',        'timestamp',  ''),
    ('AuditPipelineRunId',        'string',  ''),
    ('AuditFilePath',        'string',  ''),
    ('RevisionHash',        'binary',  ''),
    ]

# Combine base and audit columns
all_columns = base_columns + audit_columns


comparisonArray = [
    {
        "col_name":    col,
        "data_type":   dt,
        "comment":     comment
    }
    for col, dt, comment in all_columns
]

partitions = []

# Table alias and path
table_alias = "part_bom"
path = destination_filepath + table_alias

# Invoke ProcessTable
ProcessTable(
    "LH_BI_SILVER_TIER1",
    table_alias,
    comparisonArray,
    partitions,
    path
)


StatementMeta(, e00e84cf-a8f6-49bf-a364-9470ed7caf20, 19, Finished, Available, Finished, False)

## process shipper_container

In [18]:


# Provide Table-specific columns: (name, data_type, comment)
base_columns = [
('PCN','INT',''),
('Shipper_Line_Key','INT',''),
('Serial_No','STRING',''),
('Release_Key','INT',''),
('Quantity','DECIMAL(16,4)',''),
('Loaded_Date','timestamp',''),
('Shipper_Container_Key','INT',''),
]

# Audit-prefixed columns to include for every table 
audit_columns = [ 
    ('Audit_SourceFileOrFolder',      'string',     ''), 
    ('Audit_CreatedDateTime',         'timestamp',  ''), 
    ('Audit_ModifiedDateTime',        'timestamp',  ''),
    ('Audit_Source_ModifiedDateTime',        'timestamp',  ''),
    ('AuditPipelineTriggerTime',        'timestamp',  ''),
    ('AuditPipelineRunId',        'string',  ''),
    ('AuditFilePath',        'string',  ''),
    ('RevisionHash',        'binary',  ''),
    ]

# Combine base and audit columns
all_columns = base_columns + audit_columns


comparisonArray = [
    {
        "col_name":    col,
        "data_type":   dt,
        "comment":     comment
    }
    for col, dt, comment in all_columns
]

partitions = []

# Table alias and path
table_alias = "shipper_container"
path = destination_filepath + table_alias

# Invoke ProcessTable
ProcessTable(
    "LH_BI_SILVER_TIER1",
    table_alias,
    comparisonArray,
    partitions,
    path
)


StatementMeta(, e00e84cf-a8f6-49bf-a364-9470ed7caf20, 20, Finished, Available, Finished, False)

## Process Shipper_line

In [19]:


# Provide Table-specific columns: (name, data_type, comment)
base_columns = [
('PCN','INT',''),
('Shipper_Line_Key','INT',''),
('Shipper_Key','INT',''),
('Part_Key','INT',''),
('Quantity','decimal(16,4)',''),
('Price','decimal(16,4)',''),
('Customer_Part_Key','INT',''),
]

# Audit-prefixed columns to include for every table 
audit_columns = [ 
    ('Audit_SourceFileOrFolder',      'string',     ''), 
    ('Audit_CreatedDateTime',         'timestamp',  ''), 
    ('Audit_ModifiedDateTime',        'timestamp',  ''),
    ('Audit_Source_ModifiedDateTime',        'timestamp',  ''),
    ('AuditPipelineTriggerTime',        'timestamp',  ''),
    ('AuditPipelineRunId',        'string',  ''),
    ('AuditFilePath',        'string',  ''),
    ('RevisionHash',        'binary',  ''),
    ]

# Combine base and audit columns
all_columns = base_columns + audit_columns


comparisonArray = [
    {
        "col_name":    col,
        "data_type":   dt,
        "comment":     comment
    }
    for col, dt, comment in all_columns
]

partitions = []

# Table alias and path
table_alias = "shipper_line"
path = destination_filepath + table_alias

# Invoke ProcessTable
ProcessTable(
    "LH_BI_SILVER_TIER1",
    table_alias,
    comparisonArray,
    partitions,
    path
)


StatementMeta(, e00e84cf-a8f6-49bf-a364-9470ed7caf20, 21, Finished, Available, Finished, False)

## Process Shipper

In [20]:


# Provide Table-specific columns: (name, data_type, comment)
base_columns = [
('PCN','INT',''),
('Shipper_Key','INT',''),
('Shipper_No','string',''),
('Customer_No','int',''),
('Customer_Address_No','int',''),
('Ship_Date','timestamp',''),
('Add_Date','timestamp',''),
('Updated_Date','timestamp',''),
('Ship_From','int',''),
('Bill_To','int',''),
]

# Audit-prefixed columns to include for every table 
audit_columns = [ 
    ('Audit_SourceFileOrFolder',      'string',     ''), 
    ('Audit_CreatedDateTime',         'timestamp',  ''), 
    ('Audit_ModifiedDateTime',        'timestamp',  ''),
    ('Audit_Source_ModifiedDateTime',        'timestamp',  ''),
    ('AuditPipelineTriggerTime',        'timestamp',  ''),
    ('AuditPipelineRunId',        'string',  ''),
    ('AuditFilePath',        'string',  ''),
    ('RevisionHash',        'binary',  ''),
    ]

# Combine base and audit columns
all_columns = base_columns + audit_columns


comparisonArray = [
    {
        "col_name":    col,
        "data_type":   dt,
        "comment":     comment
    }
    for col, dt, comment in all_columns
]

partitions = []

# Table alias and path
table_alias = "shipper"
path = destination_filepath + table_alias

# Invoke ProcessTable
ProcessTable(
    "LH_BI_SILVER_TIER1",
    table_alias,
    comparisonArray,
    partitions,
    path
)


StatementMeta(, e00e84cf-a8f6-49bf-a364-9470ed7caf20, 22, Finished, Available, Finished, False)

## Process part_master

In [21]:


# Provide Table-specific columns: (name, data_type, comment)
base_columns = [
('Plexus_Customer_No','int',''),
('part_key','int',''),
('part_no','string',''),
('name','string',''),
('revision','string',''),
('part_Type','string',''),
('part_Status','string',''),
('unit','string',''),
('building_key','int',''),
('Side_Key','string',''),
('Part_group_key','int',''),
('Product_type_key','int',''),
('Part_source_key','int',''),
]

# Audit-prefixed columns to include for every table 
audit_columns = [ 
    ('Audit_SourceFileOrFolder',      'string',     ''), 
    ('Audit_CreatedDateTime',         'timestamp',  ''), 
    ('Audit_ModifiedDateTime',        'timestamp',  ''),
    ('Audit_Source_ModifiedDateTime',        'timestamp',  ''),
    ('AuditPipelineTriggerTime',        'timestamp',  ''),
    ('AuditPipelineRunId',        'string',  ''),
    ('AuditFilePath',        'string',  ''),
    ('RevisionHash',        'binary',  ''),
    ]

# Combine base and audit columns
all_columns = base_columns + audit_columns


comparisonArray = [
    {
        "col_name":    col,
        "data_type":   dt,
        "comment":     comment
    }
    for col, dt, comment in all_columns
]

partitions = []

# Table alias and path
table_alias = "part_master"
path = destination_filepath + table_alias

# Invoke ProcessTable
ProcessTable(
    "LH_BI_SILVER_TIER1",
    table_alias,
    comparisonArray,
    partitions,
    path
)


StatementMeta(, e00e84cf-a8f6-49bf-a364-9470ed7caf20, 23, Finished, Available, Finished, False)

# Process Exploded_BOM

In [22]:


# Provide Table-specific columns: (name, data_type, comment)
base_columns = [
('PCN','int',''),
('Version','String',''),
('cost_model_key','int',''),
('Child_Key','int',''),
('Part_Operation_Key','int',''),
('Parent_Part_Key','int',''),
('parent_part_no','string',''),
('Description','string',''),
('Revision','string',''),
('Name','string',''),
('Operation_No','int',''),
('Operation_Code','string',''),
('Cost_Sub_Type_Key','int',''),
('Cost_Type','string',''),
('Cost_Sub_Type','string',''),
('Quantity','decimal(16,6)',''),
('Cost','decimal(16,6)',''),
('Level','int',''),
('Path','string',''),
('CT_Sort_Order','int',''),
('CST_Sort_Order','int',''),
('PartRowSpan','int',''),
('OpRowSpan','int',''),
('Note','string',''),
('Calc_Note','string',''),
('zero_level_desc','string',''),

]

# Audit-prefixed columns to include for every table
audit_columns = [ 
    ('Audit_SourceFileOrFolder',      'string',     ''), 
    ('Audit_CreatedDateTime',         'timestamp',  ''), 
    ('Audit_ModifiedDateTime',        'timestamp',  ''),
    ('Audit_Source_ModifiedDateTime',        'timestamp',  ''),
    ('AuditPipelineTriggerTime',        'timestamp',  ''),
    ('AuditPipelineRunId',        'string',  ''),
    ('AuditFilePath',        'string',  ''),
    ('RevisionHash',        'binary',  ''),
    ]

# Combine base and audit columns
all_columns = base_columns + audit_columns


comparisonArray = [
    {
        "col_name":    col,
        "data_type":   dt,
        "comment":     comment
    }
    for col, dt, comment in all_columns
]

partitions = []

# Table alias and path
table_alias = "plex_exploded_bom"
path = destination_filepath + table_alias

# Invoke ProcessTable
ProcessTable(
    "LH_BI_SILVER_TIER1",
    table_alias,
    comparisonArray,
    partitions,
    path
)


StatementMeta(, e00e84cf-a8f6-49bf-a364-9470ed7caf20, 24, Finished, Available, Finished, False)

#### Process table commodity.

In [23]:
comparisonArray=[
	{"col_name":"PCN", "data_type":"int","comment":"[ID:{29615}]"}
	,{"col_name":"Building", "data_type":"string","comment":"[ID:{84932}]"}
	,{"col_name":"Part_No", "data_type":"string","comment":"[ID:{73045}]"}
	,{"col_name":"Commodity", "data_type":"string","comment":"[ID:{44383}]"}
	,{"col_name":"Sub_Commodity", "data_type":"string","comment":"[ID:{50616}]"}
	,{"col_name":"AuditPipelineTriggerTime", "data_type":"date","comment":"[ID:{59935}]"}
	,{"col_name":"AuditPipelineRunId", "data_type":"string","comment":"[ID:{24411}]"}
	,{"col_name":"AuditFilePath", "data_type":"string","comment":"[ID:{96427}]"}
	,{"col_name":"Audit_Source_ModifiedDateTime", "data_type":"timestamp","comment":""}
	,{"col_name":"RevisionHash", "data_type":"binary","comment":""}
	,{"col_name":"Audit_SourceFileOrFolder", "data_type":"string","comment":""}
	,{"col_name":"Audit_CreatedDateTime", "data_type":"timestamp","comment":""}
	,{"col_name":"Audit_ModifiedDateTime", "data_type":"timestamp","comment":""}
]
partitions=[]
path=destination_filepath+"commodity"

ProcessTable("LH_BI_SILVER_TIER1","commodity",comparisonArray,partitions,path)

StatementMeta(, e00e84cf-a8f6-49bf-a364-9470ed7caf20, 25, Finished, Available, Finished, False)

In [24]:
%%sql
ALTER TABLE LH_BI_SILVER_TIER1.commodity DROP CONSTRAINT IF EXISTS PCNnotNull;--non "STOP" constraints are not created as check constraints.

StatementMeta(, e00e84cf-a8f6-49bf-a364-9470ed7caf20, 26, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

#### Process table line_item.

In [25]:
comparisonArray=[
	{"col_name":"PCN", "data_type":"string","comment":"[ID:{59021}]"}
	,{"col_name":"Line_Item_Key", "data_type":"int","comment":"[ID:{30109}]"}
	,{"col_name":"Line_Item_No", "data_type":"string","comment":"[ID:{75106}]"}
	,{"col_name":"Supplier_Part_No", "data_type":"string","comment":"[ID:{43811}]"}
	,{"col_name":"Unit", "data_type":"string","comment":"[ID:{36717}]"}
	,{"col_name":"Unit_Price", "data_type":"string","comment":"[ID:{48936}]"}
	,{"col_name":"part_key", "data_type":"int","comment":"[ID:{11031}]"}
	,{"col_name":"PO_Key", "data_type":"int","comment":"[ID:{72452}]"}
	,{"col_name":"item_Key", "data_type":"int","comment":"[ID:{94632}]"}
	,{"col_name":"Order_Unit", "data_type":"string","comment":"[ID:{45186}]"}
	,{"col_name":"effective_Date", "data_type":"string","comment":"[ID:{26763}]"}
	,{"col_name":"Expiration_date", "data_type":"string","comment":"[ID:{98244}]"}
	,{"col_name":"Add_Date", "data_type":"string","comment":"[ID:{58420}]"}
	,{"col_name":"MAster_price_Use", "data_type":"string","comment":"[ID:{95319}]"}
	,{"col_name":"Line_Item_Category_Key", "data_type":"string","comment":"[ID:{76675}]"}
	,{"col_name":"AuditPipelineTriggerTime", "data_type":"date","comment":"[ID:{59935}]"}
	,{"col_name":"AuditPipelineRunId", "data_type":"string","comment":"[ID:{24411}]"}
	,{"col_name":"AuditFilePath", "data_type":"string","comment":"[ID:{96427}]"}
	,{"col_name":"Audit_Source_ModifiedDateTime", "data_type":"timestamp","comment":""}
	,{"col_name":"RevisionHash", "data_type":"binary","comment":""}
	,{"col_name":"Audit_SourceFileOrFolder", "data_type":"string","comment":""}
	,{"col_name":"Audit_CreatedDateTime", "data_type":"timestamp","comment":""}
	,{"col_name":"Audit_ModifiedDateTime", "data_type":"timestamp","comment":""}
]
partitions=[]
path=destination_filepath+"line_item"

ProcessTable("LH_BI_SILVER_TIER1","line_item",comparisonArray,partitions,path)

StatementMeta(, e00e84cf-a8f6-49bf-a364-9470ed7caf20, 27, Finished, Available, Finished, False)

#### Process  part

In [26]:
comparisonArray=[
	{"col_name":"PCN", "data_type":"string","comment":"[ID:{45858}]"}
	,{"col_name":"Part_key", "data_type":"string","comment":"[ID:{55906}]"}
	,{"col_name":"Part_No", "data_type":"string","comment":"[ID:{50088}]"}
	,{"col_name":"Name", "data_type":"string","comment":"[ID:{88586}]"}
	,{"col_name":"revision", "data_type":"string","comment":"[ID:{31439}]"}
	,{"col_name":"Part_type", "data_type":"string","comment":"[ID:{77942}]"}
	,{"col_name":"Part_status", "data_type":"string","comment":"[ID:{28765}]"}
	,{"col_name":"Unit", "data_type":"string","comment":"[ID:{73186}]"}
	,{"col_name":"Building_key", "data_type":"string","comment":"[ID:{83133}]"}
	,{"col_name":"Side_Key", "data_type":"string","comment":"[ID:{83134}]"}
	,{"col_name":"Description", "data_type":"string","comment":"[ID:{41116}]"}
	,{"col_name":"AuditPipelineTriggerTime", "data_type":"date","comment":"[ID:{59935}]"}
	,{"col_name":"AuditPipelineRunId", "data_type":"string","comment":"[ID:{24411}]"}
	,{"col_name":"AuditFilePath", "data_type":"string","comment":"[ID:{96427}]"}
	,{"col_name":"Audit_Source_ModifiedDateTime", "data_type":"timestamp","comment":""}
	,{"col_name":"RevisionHash", "data_type":"binary","comment":""}
	,{"col_name":"Audit_SourceFileOrFolder", "data_type":"string","comment":""}
	,{"col_name":"Audit_CreatedDateTime", "data_type":"timestamp","comment":""}
	,{"col_name":"Audit_ModifiedDateTime", "data_type":"timestamp","comment":""}
]
partitions=[]
path=destination_filepath+"part"

ProcessTable("LH_BI_SILVER_TIER1","part",comparisonArray,partitions,path)

StatementMeta(, e00e84cf-a8f6-49bf-a364-9470ed7caf20, 28, Finished, Available, Finished, False)

#### Process table period.

In [27]:
comparisonArray=[
	{"col_name":"PCN", "data_type":"int","comment":"[ID:{66695}]"}
	,{"col_name":"Period", "data_type":"string","comment":"[ID:{23481}]"}
	,{"col_name":"Period_Status", "data_type":"string","comment":"[ID:{85447}]"}
	,{"col_name":"Display", "data_type":"string","comment":"[ID:{41390}]"}
	,{"col_name":"Begin_Date", "data_type":"string","comment":"[ID:{71918}]"}
	,{"col_name":"End_Date", "data_type":"string","comment":"[ID:{58101}]"}
	,{"col_name":"Period_Display", "data_type":"string","comment":"[ID:{53697}]"}
	,{"col_name":"Fiscal_Order", "data_type":"smallint","comment":"[ID:{36715}]"}
	,{"col_name":"Period_Key", "data_type":"int","comment":"[ID:{27003}]"}
	,{"col_name":"Quarter_Group", "data_type":"smallint","comment":"[ID:{98875}]"}
	,{"col_name":"Add_Date", "data_type":"string","comment":"[ID:{41756}]"}
	,{"col_name":"AuditPipelineTriggerTime", "data_type":"date","comment":"[ID:{59935}]"}
	,{"col_name":"AuditPipelineRunId", "data_type":"string","comment":"[ID:{24411}]"}
	,{"col_name":"AuditFilePath", "data_type":"string","comment":"[ID:{96427}]"}
	,{"col_name":"Audit_Source_ModifiedDateTime", "data_type":"timestamp","comment":""}
	,{"col_name":"RevisionHash", "data_type":"binary","comment":""}
	,{"col_name":"Audit_SourceFileOrFolder", "data_type":"string","comment":""}
	,{"col_name":"Audit_CreatedDateTime", "data_type":"timestamp","comment":""}
	,{"col_name":"Audit_ModifiedDateTime", "data_type":"timestamp","comment":""}
]
partitions=[]
path=destination_filepath+"period"

ProcessTable("LH_BI_SILVER_TIER1","period",comparisonArray,partitions,path)

StatementMeta(, e00e84cf-a8f6-49bf-a364-9470ed7caf20, 29, Finished, Available, Finished, False)

#### Process table po.

In [28]:
comparisonArray=[
	{"col_name":"PCN", "data_type":"int","comment":"[ID:{96024}]"}
	,{"col_name":"PO_No", "data_type":"string","comment":"[ID:{63927}]"}
	,{"col_name":"PO_Revision", "data_type":"string","comment":"[ID:{97429}]"}
	,{"col_name":"PO_Type", "data_type":"string","comment":"[ID:{36877}]"}
	,{"col_name":"Supplier_No", "data_type":"int","comment":"[ID:{51695}]"}
	,{"col_name":"Terms", "data_type":"string","comment":"[ID:{52918}]"}
	,{"col_name":"Blanket_Order", "data_type":"smallint","comment":"[ID:{17953}]"}
	,{"col_name":"PO_Ship_To", "data_type":"string","comment":"[ID:{87046}]"}
	,{"col_name":"PO_Date", "data_type":"string","comment":"[ID:{28057}]"}
	,{"col_name":"Issued_By", "data_type":"int","comment":"[ID:{70246}]"}
	,{"col_name":"Due_Date", "data_type":"string","comment":"[ID:{70928}]"}
	,{"col_name":"PO_Key", "data_type":"int","comment":"[ID:{46457}]"}
	,{"col_name":"Building_key", "data_type":"string","comment":"[ID:{80326}]"}
	,{"col_name":"Currency_Code", "data_type":"string","comment":"[ID:{77658}]"}
	,{"col_name":"PO_Status_Key", "data_type":"int","comment":"[ID:{16234}]"}
	,{"col_name":"PO_Type_Key", "data_type":"int","comment":"[ID:{43441}]"}
	,{"col_name":"PO_Category_Key", "data_type":"int","comment":"[ID:{62857}]"}
	,{"col_name":"INCO_Terms_Key", "data_type":"int","comment":"[ID:{74556}]"}
	,{"col_name":"AuditPipelineTriggerTime", "data_type":"date","comment":"[ID:{59935}]"}
	,{"col_name":"AuditPipelineRunId", "data_type":"string","comment":"[ID:{24411}]"}
	,{"col_name":"AuditFilePath", "data_type":"string","comment":"[ID:{96427}]"}
	,{"col_name":"Audit_Source_ModifiedDateTime", "data_type":"timestamp","comment":""}
	,{"col_name":"RevisionHash", "data_type":"binary","comment":""}
	,{"col_name":"Audit_SourceFileOrFolder", "data_type":"string","comment":""}
	,{"col_name":"Audit_CreatedDateTime", "data_type":"timestamp","comment":""}
	,{"col_name":"Audit_ModifiedDateTime", "data_type":"timestamp","comment":""}
]
partitions=[]
path=destination_filepath+"po"

ProcessTable("LH_BI_SILVER_TIER1","po",comparisonArray,partitions,path)

StatementMeta(, e00e84cf-a8f6-49bf-a364-9470ed7caf20, 30, Finished, Available, Finished, False)

#### Process table receipt.

In [29]:
comparisonArray=[
	{"col_name":"PCN", "data_type":"int","comment":"[ID:{54343}]"}
	,{"col_name":"receipt_key", "data_type":"int","comment":"[ID:{49180}]"}
	,{"col_name":"line_item_key", "data_type":"int","comment":"[ID:{88246}]"}
	,{"col_name":"release_no", "data_type":"int","comment":"[ID:{65228}]"}
	,{"col_name":"receive_Date", "data_type":"string","comment":"[ID:{73563}]"}
	,{"col_name":"quantity", "data_type":"string","comment":"[ID:{83457}]"}
	,{"col_name":"supplier_no", "data_type":"int","comment":"[ID:{39974}]"}
	,{"col_name":"exchange_rate", "data_type":"string","comment":"[ID:{70010}]"}
	,{"col_name":"unit_price", "data_type":"string","comment":"[ID:{89827}]"}
	,{"col_name":"Price_unit", "data_type":"string","comment":"[ID:{87336}]"}
	,{"col_name":"order_unit", "data_type":"string","comment":"[ID:{47080}]"}
	,{"col_name":"part_key", "data_type":"int","comment":"[ID:{38548}]"}
	,{"col_name":"part_operation_key", "data_type":"int","comment":"[ID:{22401}]"}
	,{"col_name":"operation_key", "data_type":"int","comment":"[ID:{97006}]"}
	,{"col_name":"inventory_Receipt_key", "data_type":"int","comment":"[ID:{31302}]"}
	,{"col_name":"created_date", "data_type":"string","comment":"[ID:{29441}]"}
	,{"col_name":"AuditPipelineTriggerTime", "data_type":"date","comment":"[ID:{59935}]"}
	,{"col_name":"AuditPipelineRunId", "data_type":"string","comment":"[ID:{24411}]"}
	,{"col_name":"AuditFilePath", "data_type":"string","comment":"[ID:{96427}]"}
	,{"col_name":"Audit_Source_ModifiedDateTime", "data_type":"timestamp","comment":""}
	,{"col_name":"RevisionHash", "data_type":"binary","comment":""}
	,{"col_name":"Audit_SourceFileOrFolder", "data_type":"string","comment":""}
	,{"col_name":"Audit_CreatedDateTime", "data_type":"timestamp","comment":""}
	,{"col_name":"Audit_ModifiedDateTime", "data_type":"timestamp","comment":""}
]
partitions=[]
path=destination_filepath+"receipt"

ProcessTable("LH_BI_SILVER_TIER1","receipt",comparisonArray,partitions,path)

StatementMeta(, e00e84cf-a8f6-49bf-a364-9470ed7caf20, 31, Finished, Available, Finished, False)

#### Process table supplier.

In [30]:
comparisonArray=[
	{"col_name":"PCN", "data_type":"int","comment":"[ID:{92320}]"}
	,{"col_name":"Supplier_No", "data_type":"int","comment":"[ID:{94242}]"}
	,{"col_name":"Supplier_Code", "data_type":"string","comment":"[ID:{33711}]"}
	,{"col_name":"Name", "data_type":"string","comment":"[ID:{19267}]"}
	,{"col_name":"Supplier_Status", "data_type":"string","comment":"[ID:{73019}]"}
	,{"col_name":"Supplier_Type", "data_type":"string","comment":"[ID:{52122}]"}
	,{"col_name":"Address", "data_type":"string","comment":"[ID:{33119}]"}
	,{"col_name":"City", "data_type":"string","comment":"[ID:{54553}]"}
	,{"col_name":"State", "data_type":"string","comment":"[ID:{40356}]"}
	,{"col_name":"Zip", "data_type":"string","comment":"[ID:{82508}]"}
	,{"col_name":"Country", "data_type":"string","comment":"[ID:{23382}]"}
	,{"col_name":"Terms", "data_type":"string","comment":"[ID:{27860}]"}
	,{"col_name":"AuditPipelineTriggerTime", "data_type":"date","comment":"[ID:{59935}]"}
	,{"col_name":"AuditPipelineRunId", "data_type":"string","comment":"[ID:{24411}]"}
	,{"col_name":"AuditFilePath", "data_type":"string","comment":"[ID:{96427}]"}
	,{"col_name":"Audit_Source_ModifiedDateTime", "data_type":"timestamp","comment":""}
	,{"col_name":"RevisionHash", "data_type":"binary","comment":""}
	,{"col_name":"Audit_SourceFileOrFolder", "data_type":"string","comment":""}
	,{"col_name":"Audit_CreatedDateTime", "data_type":"timestamp","comment":""}
	,{"col_name":"Audit_ModifiedDateTime", "data_type":"timestamp","comment":""}
]
partitions=[]
path=destination_filepath+"supplier"

ProcessTable("LH_BI_SILVER_TIER1","supplier",comparisonArray,partitions,path)

StatementMeta(, e00e84cf-a8f6-49bf-a364-9470ed7caf20, 32, Finished, Available, Finished, False)

#### Process Table AR_Inv_Details

In [31]:
comparisonArray=[
{"col_name":"Plexus_Customer_No","data_type":"int","comment":"[ID:{92319}]"}
,{"col_name":"Plexus_Customer_Name","data_type":"string","comment":"[ID:{92320}]"}
,{"col_name":"Customer_No","data_type":"int","comment":"[ID:{92321}]"}
,{"col_name":"Customer_Code","data_type":"string","comment":"[ID:{94242}]"}
,{"col_name":"other_Customer_Code","data_type":"string","comment":"[ID:{33711}]"}
,{"col_name":"Ship_To_Note","data_type":"string","comment":"[ID:{19267}]"}
,{"col_name":"Invoice_Link","data_type":"int","comment":"[ID:{33710}]"}
,{"col_name":"Invoice_no","data_type":"string","comment":"[ID:{33712}]"}
,{"col_name":"Customer_PO_No","data_type":"string","comment":"[ID:{33713}]"}
,{"col_name":"Period","data_type":"string","comment":"[ID:{73019}]"}
,{"col_name":"Invoice_Date","data_type":"string","comment":"[ID:{52122}]"}
,{"col_name":"Supplier_No","data_type":"int","comment":"[ID:{52123}]"}
,{"col_name":"Carrier","data_type":"string","comment":"[ID:{33714}]"}
,{"col_name":"Shipper_No","data_type":"int","comment":"[ID:{33119}}]"}
,{"col_name":"Old_Part_No","data_type":"string","comment":"[ID:{54553}]"}
,{"col_name":"Order_Quantity","data_type":"string","comment":"[ID:{40356}]"}
,{"col_name":"Ship_Date","data_type":"string","comment":"[ID:{82508}]"}
,{"col_name":"Invoice_Amount","data_type":"string","comment":"[ID:{23382}]"}
,{"col_name":"Bill_To","data_type":"string","comment":"[ID:{33727}]"}
,{"col_name":"Ship_To","data_type":"string","comment":"[ID:{27860}]"}
,{"col_name":"Shipper_Key","data_type":"int","comment":"[ID:{27861}]"}
,{"col_name":"Part_key","data_type":"int","comment":"[ID:{27862}]"}
,{"col_name":"Part_No","data_type":"string","comment":"[ID:{33728}]"}
,{"col_name":"Order_No","data_type":"string","comment":"[ID:{33716}]"}
,{"col_name":"Part_Internal_Note","data_type":"string","comment":"[ID:{33717}]"}
,{"col_name":"PO_No","data_type":"string","comment":"[ID:{33718}]"}
,{"col_name":"Account_No","data_type":"string","comment":"[ID:{33719}]"}
,{"col_name":"line_Item_No","data_type":"string","comment":"[ID:{33720}]"}
,{"col_name":"Quantity","data_type":"string","comment":"[ID:{33721}]"}
,{"col_name":"Unit_Price","data_type":"string","comment":"[ID:{33722}]"}
,{"col_name":"Currency_amount","data_type":"string","comment":"[ID:{33723}]"}
,{"col_name":"Customer_Part_No","data_type":"string","comment":"[ID:{33724}]"}
,{"col_name":"Currency_code","data_type":"string","comment":"[ID:{33725}]"}
,{"col_name":"Exchange_Rate","data_type":"string","comment":"[ID:{33726}]"}
,{"col_name":"Bill_to_address","data_type":"int","comment":"[ID:{33729}]"}
,{"col_name":"Ship_to_address","data_type":"int","comment":"[ID:{33730}]"}
,{"col_name":"AuditPipelineTriggerTime", "data_type":"date","comment":"[ID:{59935}]"}
,{"col_name":"AuditPipelineRunId", "data_type":"string","comment":"[ID:{24411}]"}
,{"col_name":"AuditFilePath", "data_type":"string","comment":"[ID:{96427}]"}
,{"col_name":"Audit_Source_ModifiedDateTime", "data_type":"timestamp","comment":""}
,{"col_name":"RevisionHash", "data_type":"binary","comment":""}
,{"col_name":"Audit_SourceFileOrFolder", "data_type":"string","comment":""}
,{"col_name":"Audit_CreatedDateTime", "data_type":"timestamp","comment":""}
,{"col_name":"Audit_ModifiedDateTime", "data_type":"timestamp","comment":""}
]
partitions=[]
path=destination_filepath+"AR_Inv_Details"

ProcessTable("LH_BI_SILVER_TIER1","AR_Inv_Details",comparisonArray,partitions,path)

StatementMeta(, e00e84cf-a8f6-49bf-a364-9470ed7caf20, 33, Finished, Available, Finished, False)

#### Process Table Customer_Address

In [32]:
comparisonArray=[
	{"col_name":"PCN", "data_type":"int","comment":"[ID:{29615}]"}
	,{"col_name":"Customer_No", "data_type":"int","comment":"[ID:{84932}]"}
	,{"col_name":"Customer_Address_No", "data_type":"int","comment":"[ID:{73045}]"}
	,{"col_name":"Customer_Address_Code", "data_type":"string","comment":"[ID:{44383}]"}
	,{"col_name":"Address", "data_type":"string","comment":"[ID:{50616}]"}
    ,{"col_name":"City", "data_type":"string","comment":"[ID:{50617}]"}
    ,{"col_name":"State", "data_type":"string","comment":"[ID:{50618}]"}
    ,{"col_name":"Zip", "data_type":"string","comment":"[ID:{50619}]"}
	,{"col_name":"AuditPipelineTriggerTime", "data_type":"date","comment":"[ID:{59935}]"}
	,{"col_name":"AuditPipelineRunId", "data_type":"string","comment":"[ID:{24411}]"}
	,{"col_name":"AuditFilePath", "data_type":"string","comment":"[ID:{96427}]"}
	,{"col_name":"Audit_Source_ModifiedDateTime", "data_type":"timestamp","comment":""}
	,{"col_name":"RevisionHash", "data_type":"binary","comment":""}
	,{"col_name":"Audit_SourceFileOrFolder", "data_type":"string","comment":""}
	,{"col_name":"Audit_CreatedDateTime", "data_type":"timestamp","comment":""}
	,{"col_name":"Audit_ModifiedDateTime", "data_type":"timestamp","comment":""}
]
partitions=[]
path=destination_filepath+"customer_address"

ProcessTable("LH_BI_SILVER_TIER1","customer_address",comparisonArray,partitions,path)

StatementMeta(, e00e84cf-a8f6-49bf-a364-9470ed7caf20, 34, Finished, Available, Finished, False)

#### Process Table Customer

In [33]:
comparisonArray=[
	{"col_name":"PCN", "data_type":"int","comment":"[ID:{29615}]"}
	,{"col_name":"Customer_No", "data_type":"int","comment":"[ID:{84932}]"}
	,{"col_name":"Customer_Code", "data_type":"string","comment":"[ID:{73045}]"}
	,{"col_name":"Name", "data_type":"string","comment":"[ID:{44383}]"}
	,{"col_name":"Customer_Status", "data_type":"string","comment":"[ID:{50616}]"}
    ,{"col_name":"Customer_Type", "data_type":"string","comment":"[ID:{50617}]"}
    ,{"col_name":"Terms", "data_type":"string","comment":"[ID:{50618}]"}
  	,{"col_name":"AuditPipelineTriggerTime", "data_type":"date","comment":"[ID:{59935}]"}
	,{"col_name":"AuditPipelineRunId", "data_type":"string","comment":"[ID:{24411}]"}
	,{"col_name":"AuditFilePath", "data_type":"string","comment":"[ID:{96427}]"}
	,{"col_name":"Audit_Source_ModifiedDateTime", "data_type":"timestamp","comment":""}
	,{"col_name":"RevisionHash", "data_type":"binary","comment":""}
	,{"col_name":"Audit_SourceFileOrFolder", "data_type":"string","comment":""}
	,{"col_name":"Audit_CreatedDateTime", "data_type":"timestamp","comment":""}
	,{"col_name":"Audit_ModifiedDateTime", "data_type":"timestamp","comment":""}
]
partitions=[]
path=destination_filepath+"customer"

ProcessTable("LH_BI_SILVER_TIER1","customer",comparisonArray,partitions,path)

StatementMeta(, e00e84cf-a8f6-49bf-a364-9470ed7caf20, 35, Finished, Available, Finished, False)

#### Process table Currency

In [34]:
comparisonArray=[
    {"col_name":"Currency_Key", "data_type":"int","comment":"[ID:{20804}]"}
    ,{"col_name":"Currency_Symbol", "data_type":"string","comment":"[ID:{78090}]"}
    ,{"col_name":"Currency_HTML", "data_type":"string","comment":"[ID:{42531}]"}
    ,{"col_name":"Exchange_Rate", "data_type":"double","comment":"[ID:{43437}]"}
    ,{"col_name":"Exchange_Rate_Date", "data_type":"string","comment":"[ID:{3108}]"}
    ,{"col_name":"Currency_Code", "data_type":"string","comment":"[ID:{14452}]"}
    ,{"col_name":"Description", "data_type":"string","comment":"[ID:{27721}]"}
    ,{"col_name":"Sort_Order", "data_type":"int","comment":"[ID:{89167}]"}
    ,{"col_name":"Update_Date", "data_type":"string","comment":"[ID:{48923}]"}
    ,{"col_name":"Active", "data_type":"boolean","comment":"[ID:{7139}]"}
    ,{"col_name":"Currency_Unicode", "data_type":"string","comment":"[ID:{57636}]"}
  	,{"col_name":"AuditPipelineTriggerTime", "data_type":"date","comment":"[ID:{30231}]"}
	,{"col_name":"AuditPipelineRunId", "data_type":"string","comment":"[ID:{62840}]"}
	,{"col_name":"AuditFilePath", "data_type":"string","comment":"[ID:{62216}]"}
	,{"col_name":"Audit_Source_ModifiedDateTime", "data_type":"timestamp","comment":""}
	,{"col_name":"RevisionHash", "data_type":"binary","comment":""}
	,{"col_name":"Audit_SourceFileOrFolder", "data_type":"string","comment":""}
	,{"col_name":"Audit_CreatedDateTime", "data_type":"timestamp","comment":""}
	,{"col_name":"Audit_ModifiedDateTime", "data_type":"timestamp","comment":""}
]
partitions=[]
path=destination_filepath+"Currency"

ProcessTable("LH_BI_SILVER_TIER1","Currency",comparisonArray,partitions,path)

StatementMeta(, e00e84cf-a8f6-49bf-a364-9470ed7caf20, 36, Finished, Available, Finished, False)

## Process Cost_model

In [6]:
comparisonArray=[
    {"col_name":"PCN", "data_type":"int","comment":"[ID:{20804}]"}
    ,{"col_name":"Cost_Model_Key", "data_type":"string","comment":"[ID:{78090}]"}
    ,{"col_name":"Cost_Model", "data_type":"string","comment":"[ID:{42531}]"}
    ,{"col_name":"Description", "data_type":"string","comment":"[ID:{43437}]"}
    ,{"col_name":"Active", "data_type":"string","comment":"[ID:{3108}]"}
    ,{"col_name":"Primary_model", "data_type":"string","comment":"[ID:{14452}]"}
    ,{"col_name":"Frozen", "data_type":"string","comment":"[ID:{27721}]"}
  	,{"col_name":"AuditPipelineTriggerTime", "data_type":"date","comment":"[ID:{30231}]"}
	,{"col_name":"AuditPipelineRunId", "data_type":"string","comment":"[ID:{62840}]"}
	,{"col_name":"AuditFilePath", "data_type":"string","comment":"[ID:{62216}]"}
	,{"col_name":"Audit_Source_ModifiedDateTime", "data_type":"timestamp","comment":""}
	,{"col_name":"RevisionHash", "data_type":"binary","comment":""}
	,{"col_name":"Audit_SourceFileOrFolder", "data_type":"string","comment":""}
	,{"col_name":"Audit_CreatedDateTime", "data_type":"timestamp","comment":""}
	,{"col_name":"Audit_ModifiedDateTime", "data_type":"timestamp","comment":""}
]
partitions=[]
path=destination_filepath+"cost_model"

ProcessTable("LH_BI_SILVER_TIER1","cost_model",comparisonArray,partitions,path)

StatementMeta(, 38e9d83f-b5c6-4cd8-8530-47047403425f, 8, Finished, Available, Finished, False)

# Process Customer_part

In [6]:
comparisonArray=[
    {"col_name":"plexus_customer_no", "data_type":"int","comment":"[ID:{20804}]"}
    ,{"col_name":"Customer_part_key", "data_type":"string","comment":"[ID:{78090}]"}
    ,{"col_name":"Part_key", "data_type":"string","comment":"[ID:{42531}]"}
    ,{"col_name":"Customer_no", "data_type":"double","comment":"[ID:{43437}]"}
    ,{"col_name":"Customer_part_no", "data_type":"string","comment":"[ID:{3108}]"}
    ,{"col_name":"Customer_part_description", "data_type":"string","comment":"[ID:{14452}]"}
    ,{"col_name":"Supplier_no", "data_type":"string","comment":"[ID:{27721}]"}
    ,{"col_name":"Active", "data_type":"int","comment":"[ID:{89167}]"}
    ,{"col_name":"Add_date", "data_type":"timestamp","comment":"[ID:{48923}]"}
    ,{"col_name":"Update_date", "data_type":"timestamp","comment":"[ID:{7139}]"}
  	,{"col_name":"AuditPipelineTriggerTime", "data_type":"date","comment":"[ID:{30231}]"}
	,{"col_name":"AuditPipelineRunId", "data_type":"string","comment":"[ID:{62840}]"}
	,{"col_name":"AuditFilePath", "data_type":"string","comment":"[ID:{62216}]"}
	,{"col_name":"Audit_Source_ModifiedDateTime", "data_type":"timestamp","comment":""}
	,{"col_name":"RevisionHash", "data_type":"binary","comment":""}
	,{"col_name":"Audit_SourceFileOrFolder", "data_type":"string","comment":""}
	,{"col_name":"Audit_CreatedDateTime", "data_type":"timestamp","comment":""}
	,{"col_name":"Audit_ModifiedDateTime", "data_type":"timestamp","comment":""}
]
partitions=[]
path=destination_filepath+"customer_part"

ProcessTable("LH_BI_SILVER_TIER1","customer_part",comparisonArray,partitions,path)

StatementMeta(, 77ce72c9-6af8-43ab-a314-f11c28224ca9, 8, Finished, Available, Finished, False)

# Process Approved_workcenter

In [37]:
comparisonArray=[
    {"col_name":"Plexus_Customer_No", "data_type":"int","comment":"[ID:{20804}]"}
    ,{"col_name":"Part_Key", "data_type":"int","comment":"[ID:{78090}]"}
    ,{"col_name":"Part_Operation_Key", "data_type":"int","comment":"[ID:{42531}]"}
    ,{"col_name":"Workcenter_Key", "data_type":"int","comment":"[ID:{43437}]"}
    ,{"col_name":"Crew_Size", "data_type":"decimal(16,6)","comment":"[ID:{3108}]"}
    ,{"col_name":"Standard_Production_Rate","data_type":"decimal(16,6)","comment":"[ID:{33720}]"}
    ,{"col_name":"Sort_Order", "data_type":"int","comment":"[ID:{14452}]"}
    ,{"col_name":"Note", "data_type":"string","comment":"[ID:{27721}]"}
    ,{"col_name":"Lead_Time", "data_type":"decimal(16,6)","comment":"[ID:{89167}]"}
    ,{"col_name":"Setup_Time", "data_type":"decimal(16,6)","comment":"[ID:{48923}]"}
    ,{"col_name":"Ideal_Rate", "data_type":"decimal(16,6)","comment":"[ID:{7139}]"}
    ,{"col_name":"Target_Rate","data_type":"decimal(16,6)","comment":"[ID:{73019}]"}
    ,{"col_name":"Setup_Crew_Size","data_type":"decimal(16,6)","comment":"[ID:{52122}]"}
    ,{"col_name":"Inspector_Crew_Size","data_type":"decimal(16,6)","comment":"[ID:{52123}]"}
    ,{"col_name":"Cycle_Time","data_type":"decimal(16,6)","comment":"[ID:{33714}]"}
    ,{"col_name":"AuditPipelineTriggerTime", "data_type":"date","comment":"[ID:{30231}]"}
	,{"col_name":"AuditPipelineRunId", "data_type":"string","comment":"[ID:{62840}]"}
	,{"col_name":"AuditFilePath", "data_type":"string","comment":"[ID:{62216}]"}
	,{"col_name":"Audit_Source_ModifiedDateTime", "data_type":"timestamp","comment":""}
	,{"col_name":"RevisionHash", "data_type":"binary","comment":""}
	,{"col_name":"Audit_SourceFileOrFolder", "data_type":"string","comment":""}
	,{"col_name":"Audit_CreatedDateTime", "data_type":"timestamp","comment":""}
	,{"col_name":"Audit_ModifiedDateTime", "data_type":"timestamp","comment":""}
]
partitions=[]
path=destination_filepath+"approved_workcenter"

ProcessTable("LH_BI_SILVER_TIER1","approved_workcenter",comparisonArray,partitions,path)

StatementMeta(, e00e84cf-a8f6-49bf-a364-9470ed7caf20, 39, Finished, Available, Finished, False)

# Process cost_sub_type

In [38]:
comparisonArray=[
    {"col_name":"PCN", "data_type":"int", "comment":"[ID:{20804}]"},
    {"col_name":"Cost_Sub_Type_Key", "data_type":"int", "comment":"[ID:{78090}]"},
    {"col_name":"Cost_Sub_Type", "data_type":"STRING", "comment":"[ID:{78091}]"},
    {"col_name":"Cost_Type_Key", "data_type":"int", "comment":"[ID:{42531}]"},
    {"col_name":"Sort_Order", "data_type":"int", "comment":"[ID:{14452}]"},
    {"col_name":"Workcenter_logout", "data_type":"smallint", "comment":"[ID:{43437}]"},
    {"col_name":"Labor_Tracking", "data_type":"smallint", "comment":"[ID:{3108}]"},
    {"col_name":"Account_No", "data_type":"string", "comment":"[ID:{33719}]"},
    {"col_name":"Image", "data_type":"string", "comment":"[ID:{33709}]"},

    {"col_name":"Note", "data_type":"string", "comment":"[ID:{27721}]"},
    {"col_name":"Timeclock", "data_type":"smallint", "comment":"[ID:{89167}]"},
    {"col_name":"Production_Cost", "data_type":"smallint", "comment":"[ID:{33720}]"},
    {"col_name":"Setup_Cost", "data_type":"smallint", "comment":"[ID:{48923}]"},
    {"col_name":"Direct_Labor", "data_type":"smallint", "comment":"[ID:{7139}]"},
    {"col_name":"Indirect_Labor", "data_type":"smallint", "comment":"[ID:{73019}]"},
    {"col_name":"Use_For_Budgeting", "data_type":"smallint", "comment":"[ID:{52122}]"},
    {"col_name":"Include_In_Purchase_Price_Report", "data_type":"smallint", "comment":"[ID:{52123}]"},
    {"col_name":"Cost_Of_Goods_Sold", "data_type":"BOOLEAN", "comment":"[ID:{33714}]"},
    {"col_name":"Active","data_type":"BOOLEAN","comment":"[ID:{33730}]"},
    {"col_name":"AuditPipelineTriggerTime", "data_type":"date", "comment":"[ID:{30231}]"},
    {"col_name":"AuditPipelineRunId", "data_type":"string", "comment":"[ID:{62840}]"},
    {"col_name":"AuditFilePath", "data_type":"string", "comment":"[ID:{62216}]"},
    {"col_name":"Audit_Source_ModifiedDateTime", "data_type":"timestamp", "comment":""},
    {"col_name":"RevisionHash", "data_type":"binary", "comment":""},
    {"col_name":"Audit_SourceFileOrFolder", "data_type":"string", "comment":""},
    {"col_name":"Audit_CreatedDateTime", "data_type":"timestamp", "comment":""},
    {"col_name":"Audit_ModifiedDateTime", "data_type":"timestamp", "comment":""}
]

partitions=[]
path=destination_filepath+"cost_sub_type"

ProcessTable("LH_BI_SILVER_TIER1","cost_sub_type",comparisonArray,partitions,path)

StatementMeta(, e00e84cf-a8f6-49bf-a364-9470ed7caf20, 40, Finished, Available, Finished, False)

# Process Cost_type

In [39]:
comparisonArray=[
    {"col_name":"PCN", "data_type":"int", "comment":"[ID:{20804}]"},
    {"col_name":"Cost_Type_Key", "data_type":"int", "comment":"[ID:{78090}]"},
    {"col_name":"Cost_Type", "data_type":"STRING", "comment":"[ID:{78091}]"},
    {"col_name":"Sort_Order", "data_type":"int", "comment":"[ID:{14452}]"},
    {"col_name":"Valuation_Column", "data_type":"int", "comment":"[ID:{14453}]"},
    {"col_name":"AuditPipelineTriggerTime", "data_type":"date", "comment":"[ID:{30231}]"},
    {"col_name":"AuditPipelineRunId", "data_type":"string", "comment":"[ID:{62840}]"},
    {"col_name":"AuditFilePath", "data_type":"string", "comment":"[ID:{62216}]"},
    {"col_name":"Audit_Source_ModifiedDateTime", "data_type":"timestamp", "comment":""},
    {"col_name":"RevisionHash", "data_type":"binary", "comment":""},
    {"col_name":"Audit_SourceFileOrFolder", "data_type":"string", "comment":""},
    {"col_name":"Audit_CreatedDateTime", "data_type":"timestamp", "comment":""},
    {"col_name":"Audit_ModifiedDateTime", "data_type":"timestamp", "comment":""}
]

partitions=[]
path=destination_filepath+"cost_type"

ProcessTable("LH_BI_SILVER_TIER1","cost_type",comparisonArray,partitions,path)

StatementMeta(, e00e84cf-a8f6-49bf-a364-9470ed7caf20, 41, Finished, Available, Finished, False)

# Process Location

In [40]:
comparisonArray=[
    {"col_name":"Plexus_Customer_No", "data_type":"int","comment":"[ID:{20804}]"}
    ,{"col_name":"Location", "data_type":"string","comment":"[ID:{78090}]"}
    ,{"col_name":"Building_Key", "data_type":"int","comment":"[ID:{42531}]"}
    ,{"col_name":"Location_Type", "data_type":"string","comment":"[ID:{43437}]"}
    ,{"col_name":"Active", "data_type":"smallint","comment":"[ID:{3108}]"}
    ,{"col_name":"Location_Key","data_type":"int","comment":"[ID:{33720}]"}
    ,{"col_name":"Update_Date", "data_type":"timestamp","comment":"[ID:{14452}]"}

    ,{"col_name":"AuditPipelineTriggerTime", "data_type":"date","comment":"[ID:{30231}]"}
	,{"col_name":"AuditPipelineRunId", "data_type":"string","comment":"[ID:{62840}]"}
	,{"col_name":"AuditFilePath", "data_type":"string","comment":"[ID:{62216}]"}
	,{"col_name":"Audit_Source_ModifiedDateTime", "data_type":"timestamp","comment":""}
	,{"col_name":"RevisionHash", "data_type":"binary","comment":""}
	,{"col_name":"Audit_SourceFileOrFolder", "data_type":"string","comment":""}
	,{"col_name":"Audit_CreatedDateTime", "data_type":"timestamp","comment":""}
	,{"col_name":"Audit_ModifiedDateTime", "data_type":"timestamp","comment":""}
]
partitions=[]
path=destination_filepath+"location"

ProcessTable("LH_BI_SILVER_TIER1","location",comparisonArray,partitions,path)

StatementMeta(, e00e84cf-a8f6-49bf-a364-9470ed7caf20, 42, Finished, Available, Finished, False)

# Process_part_group

In [41]:
comparisonArray=[
    {"col_name":"Plexus_Customer_No", "data_type":"int","comment":"[ID:{20804}]"}
    ,{"col_name":"Part_Group_Key", "data_type":"int","comment":"[ID:{78090}]"}
    ,{"col_name":"Part_Group", "data_type":"string","comment":"[ID:{42531}]"}

    ,{"col_name":"AuditPipelineTriggerTime", "data_type":"date","comment":"[ID:{30231}]"}
	,{"col_name":"AuditPipelineRunId", "data_type":"string","comment":"[ID:{62840}]"}
	,{"col_name":"AuditFilePath", "data_type":"string","comment":"[ID:{62216}]"}
	,{"col_name":"Audit_Source_ModifiedDateTime", "data_type":"timestamp","comment":""}
	,{"col_name":"RevisionHash", "data_type":"binary","comment":""}
	,{"col_name":"Audit_SourceFileOrFolder", "data_type":"string","comment":""}
	,{"col_name":"Audit_CreatedDateTime", "data_type":"timestamp","comment":""}
	,{"col_name":"Audit_ModifiedDateTime", "data_type":"timestamp","comment":""}
]
partitions=[]
path=destination_filepath+"part_group"

ProcessTable("LH_BI_SILVER_TIER1","part_group",comparisonArray,partitions,path)

StatementMeta(, e00e84cf-a8f6-49bf-a364-9470ed7caf20, 43, Finished, Available, Finished, False)

 # Process_part_op_type

In [42]:
comparisonArray=[
    {"col_name":"PCN", "data_type":"int","comment":"[ID:{20804}]"}
    ,{"col_name":"Part_Op_Type_Key", "data_type":"int","comment":"[ID:{78090}]"}
    ,{"col_name":"Description", "data_type":"string","comment":"[ID:{42531}]"}
    ,{"col_name":"Standard", "data_type":"smallint","comment":"[ID:{43437}]"}
    ,{"col_name":"Test", "data_type":"smallint","comment":"[ID:{43438}]"}
    ,{"col_name":"Rework", "data_type":"smallint","comment":"[ID:{43439}]"}
    ,{"col_name":"Color", "data_type":"string","comment":"[ID:{3108}]"}
    ,{"col_name":"Default_Part_Op_Type","data_type":"smallint","comment":"[ID:{33720}]"}
    ,{"col_name":"Sort_Order", "data_type":"int","comment":"[ID:{14452}]"}
    ,{"col_name":"Include_In_MRP", "data_type":"boolean", "comment":"[ID:{33720}]"}
    ,{"col_name":"AuditPipelineTriggerTime", "data_type":"date","comment":"[ID:{30231}]"}
	,{"col_name":"AuditPipelineRunId", "data_type":"string","comment":"[ID:{62840}]"}
	,{"col_name":"AuditFilePath", "data_type":"string","comment":"[ID:{62216}]"}
	,{"col_name":"Audit_Source_ModifiedDateTime", "data_type":"timestamp","comment":""}
	,{"col_name":"RevisionHash", "data_type":"binary","comment":""}
	,{"col_name":"Audit_SourceFileOrFolder", "data_type":"string","comment":""}
	,{"col_name":"Audit_CreatedDateTime", "data_type":"timestamp","comment":""}
	,{"col_name":"Audit_ModifiedDateTime", "data_type":"timestamp","comment":""}
]
partitions=[]
path=destination_filepath+"part_op_type"

ProcessTable("LH_BI_SILVER_TIER1","part_op_type",comparisonArray,partitions,path)

StatementMeta(, e00e84cf-a8f6-49bf-a364-9470ed7caf20, 44, Finished, Available, Finished, False)

# Process_part_operation

In [43]:
comparisonArray=[
    {"col_name":"Plexus_Customer_No", "data_type":"int","comment":"[ID:{20804}]"}
    ,{"col_name":"Part_Key", "data_type":"int","comment":"[ID:{78090}]"}
    ,{"col_name":"Part_Operation_Key", "data_type":"int","comment":"[ID:{42531}]"}
    ,{"col_name":"Multiple", "data_type":"decimal(16,6)","comment":"[ID:{42531}]"}
    ,{"col_name":"Operation_No", "data_type":"int","comment":"[ID:{43437}]"}
    ,{"col_name":"Operation_Key", "data_type":"int","comment":"[ID:{3108}]"}
    ,{"col_name":"Description","data_type":"string","comment":"[ID:{33720}]"}
    ,{"col_name":"Active", "data_type":"smallint","comment":"[ID:{14452}]"}
    ,{"col_name":"Part_Op_Type_Key", "data_type":"int","comment":"[ID:{14453}]"}
    ,{"col_name":"Suboperation", "data_type":"int","comment":"[ID:{14454}]"}
    ,{"col_name":"AuditPipelineTriggerTime", "data_type":"date","comment":"[ID:{30231}]"}
	,{"col_name":"AuditPipelineRunId", "data_type":"string","comment":"[ID:{62840}]"}
	,{"col_name":"AuditFilePath", "data_type":"string","comment":"[ID:{62216}]"}
	,{"col_name":"Audit_Source_ModifiedDateTime", "data_type":"timestamp","comment":""}
	,{"col_name":"RevisionHash", "data_type":"binary","comment":""}
	,{"col_name":"Audit_SourceFileOrFolder", "data_type":"string","comment":""}
	,{"col_name":"Audit_CreatedDateTime", "data_type":"timestamp","comment":""}
	,{"col_name":"Audit_ModifiedDateTime", "data_type":"timestamp","comment":""}
]
partitions=[]
path=destination_filepath+"part_operation"

ProcessTable("LH_BI_SILVER_TIER1","part_operation",comparisonArray,partitions,path)

StatementMeta(, e00e84cf-a8f6-49bf-a364-9470ed7caf20, 45, Finished, Available, Finished, False)

# Process_part_product_type

In [44]:
comparisonArray=[
    {"col_name":"PCN", "data_type":"int","comment":"[ID:{20804}]"}
    ,{"col_name":"Product_Type_Key", "data_type":"int","comment":"[ID:{78090}]"}
    ,{"col_name":"Product_Type", "data_type":"string","comment":"[ID:{42531}]"}
    ,{"col_name":"Part_Group_Key", "data_type":"int","comment":"[ID:{43437}]"}
    ,{"col_name":"Manager", "data_type":"int","comment":"[ID:{3108}]"}
    ,{"col_name":"Product_Type_Code","data_type":"string","comment":"[ID:{33720}]"}
    ,{"col_name":"Scheduling_Sort_Order", "data_type":"int","comment":"[ID:{14452}]"}
    ,{"col_name":"In_MRP_Include", "data_type":"boolean", "comment":"[ID:{33720}]"}
    ,{"col_name":"AuditPipelineTriggerTime", "data_type":"date","comment":"[ID:{30231}]"}
	,{"col_name":"AuditPipelineRunId", "data_type":"string","comment":"[ID:{62840}]"}
	,{"col_name":"AuditFilePath", "data_type":"string","comment":"[ID:{62216}]"}
	,{"col_name":"Audit_Source_ModifiedDateTime", "data_type":"timestamp","comment":""}
	,{"col_name":"RevisionHash", "data_type":"binary","comment":""}
	,{"col_name":"Audit_SourceFileOrFolder", "data_type":"string","comment":""}
	,{"col_name":"Audit_CreatedDateTime", "data_type":"timestamp","comment":""}
	,{"col_name":"Audit_ModifiedDateTime", "data_type":"timestamp","comment":""}
]
partitions=[]
path=destination_filepath+"part_product_type"

ProcessTable("LH_BI_SILVER_TIER1","part_product_type",comparisonArray,partitions,path)

StatementMeta(, e00e84cf-a8f6-49bf-a364-9470ed7caf20, 46, Finished, Available, Finished, False)

# Process_Part_source

In [45]:
comparisonArray=[
    {"col_name":"PCN", "data_type":"int","comment":"[ID:{20804}]"}
    ,{"col_name":"Part_Source_Key", "data_type":"int","comment":"[ID:{78090}]"}
    ,{"col_name":"Part_Source", "data_type":"string","comment":"[ID:{42531}]"}
    ,{"col_name":"Purchased_Part", "data_type":"boolean","comment":"[ID:{43437}]"}
    ,{"col_name":"Manufactured_Part", "data_type":"boolean","comment":"[ID:{3108}]"}

    ,{"col_name":"AuditPipelineTriggerTime", "data_type":"date","comment":"[ID:{30231}]"}
	,{"col_name":"AuditPipelineRunId", "data_type":"string","comment":"[ID:{62840}]"}
	,{"col_name":"AuditFilePath", "data_type":"string","comment":"[ID:{62216}]"}
	,{"col_name":"Audit_Source_ModifiedDateTime", "data_type":"timestamp","comment":""}
	,{"col_name":"RevisionHash", "data_type":"binary","comment":""}
	,{"col_name":"Audit_SourceFileOrFolder", "data_type":"string","comment":""}
	,{"col_name":"Audit_CreatedDateTime", "data_type":"timestamp","comment":""}
	,{"col_name":"Audit_ModifiedDateTime", "data_type":"timestamp","comment":""}
]
partitions=[]
path=destination_filepath+"part_source"

ProcessTable("LH_BI_SILVER_TIER1","part_source",comparisonArray,partitions,path)

StatementMeta(, e00e84cf-a8f6-49bf-a364-9470ed7caf20, 47, Finished, Available, Finished, False)

# Process_Part_status

In [46]:
comparisonArray=[
    {"col_name":"Plexus_Customer_No", "data_type":"int","comment":"[ID:{20804}]"}
    ,{"col_name":"Part_Status", "data_type":"string","comment":"[ID:{78090}]"}
    ,{"col_name":"Active", "data_type":"smallint","comment":"[ID:{42531}]"}
    ,{"col_name":"Part_Status_Key", "data_type":"int","comment":"[ID:{43437}]"}

    ,{"col_name":"AuditPipelineTriggerTime", "data_type":"date","comment":"[ID:{30231}]"}
	,{"col_name":"AuditPipelineRunId", "data_type":"string","comment":"[ID:{62840}]"}
	,{"col_name":"AuditFilePath", "data_type":"string","comment":"[ID:{62216}]"}
	,{"col_name":"Audit_Source_ModifiedDateTime", "data_type":"timestamp","comment":""}
	,{"col_name":"RevisionHash", "data_type":"binary","comment":""}
	,{"col_name":"Audit_SourceFileOrFolder", "data_type":"string","comment":""}
	,{"col_name":"Audit_CreatedDateTime", "data_type":"timestamp","comment":""}
	,{"col_name":"Audit_ModifiedDateTime", "data_type":"timestamp","comment":""}
]
partitions=[]
path=destination_filepath+"part_status"

ProcessTable("LH_BI_SILVER_TIER1","part_status",comparisonArray,partitions,path)

StatementMeta(, e00e84cf-a8f6-49bf-a364-9470ed7caf20, 48, Finished, Available, Finished, False)

# Process_Part_type

In [47]:
comparisonArray=[
    {"col_name":"Plexus_Customer_No", "data_type":"int","comment":"[ID:{20804}]"}
    ,{"col_name":"Part_Type", "data_type":"string","comment":"[ID:{78090}]"}
    ,{"col_name":"Sort_Order", "data_type":"int","comment":"[ID:{42531}]"}
    ,{"col_name":"Part_Group_Key", "data_type":"int","comment":"[ID:{43437}]"}
    ,{"col_name":"Part_Type_Key", "data_type":"int","comment":"[ID:{3108}]"}
	,{"col_name":"Include_In_Standard_Cost", "data_type":"int","comment":"[ID:{3109}]"}
    ,{"col_name":"AuditPipelineTriggerTime", "data_type":"date","comment":"[ID:{30231}]"}
	,{"col_name":"AuditPipelineRunId", "data_type":"string","comment":"[ID:{62840}]"}
	,{"col_name":"AuditFilePath", "data_type":"string","comment":"[ID:{62216}]"}
	,{"col_name":"Audit_Source_ModifiedDateTime", "data_type":"timestamp","comment":""}
	,{"col_name":"RevisionHash", "data_type":"binary","comment":""}
	,{"col_name":"Audit_SourceFileOrFolder", "data_type":"string","comment":""}
	,{"col_name":"Audit_CreatedDateTime", "data_type":"timestamp","comment":""}
	,{"col_name":"Audit_ModifiedDateTime", "data_type":"timestamp","comment":""}
]
partitions=[]
path=destination_filepath+"part_type"

ProcessTable("LH_BI_SILVER_TIER1","part_type",comparisonArray,partitions,path)

StatementMeta(, e00e84cf-a8f6-49bf-a364-9470ed7caf20, 49, Finished, Available, Finished, False)

#  Process_sales_po_line

In [48]:
comparisonArray=[
    {"col_name":"PCN", "data_type":"int","comment":"[ID:{20804}]"}
    ,{"col_name":"PO_Line_Key", "data_type":"int","comment":"[ID:{78090}]"}
    ,{"col_name":"PO_Key", "data_type":"int","comment":"[ID:{42531}]"}
    ,{"col_name":"Part_Key", "data_type":"int","comment":"[ID:{43437}]"}
    ,{"col_name":"Customer_Part_Key", "data_type":"int","comment":"[ID:{3108}]"}
    ,{"col_name":"Add_Date","data_type":"timestamp","comment":"[ID:{33720}]"}
    ,{"col_name":"Update_Date", "data_type":"timestamp","comment":"[ID:{14452}]"}
    ,{"col_name":"Master_Unit_Type_Key", "data_type":"int","comment":"[ID:{27721}]"}
    ,{"col_name":"Active", "data_type":"smallint","comment":"[ID:{89167}]"}

    ,{"col_name":"AuditPipelineTriggerTime", "data_type":"date","comment":"[ID:{30231}]"}
	,{"col_name":"AuditPipelineRunId", "data_type":"string","comment":"[ID:{62840}]"}
	,{"col_name":"AuditFilePath", "data_type":"string","comment":"[ID:{62216}]"}
	,{"col_name":"Audit_Source_ModifiedDateTime", "data_type":"timestamp","comment":""}
	,{"col_name":"RevisionHash", "data_type":"binary","comment":""}
	,{"col_name":"Audit_SourceFileOrFolder", "data_type":"string","comment":""}
	,{"col_name":"Audit_CreatedDateTime", "data_type":"timestamp","comment":""}
	,{"col_name":"Audit_ModifiedDateTime", "data_type":"timestamp","comment":""}
]
partitions=[]
path=destination_filepath+"po_line"

ProcessTable("LH_BI_SILVER_TIER1","po_line",comparisonArray,partitions,path)

StatementMeta(, e00e84cf-a8f6-49bf-a364-9470ed7caf20, 50, Finished, Available, Finished, False)

# Process_Price

In [49]:
comparisonArray=[
    {"col_name":"PCN", "data_type":"int","comment":"[ID:{20804}]"}
    ,{"col_name":"Price_Key", "data_type":"int","comment":"[ID:{78090}]"}
    ,{"col_name":"PO_Line_Key", "data_type":"int","comment":"[ID:{42531}]"}
    ,{"col_name":"Effective_Date", "data_type":"timestamp","comment":"[ID:{43437}]"}
    ,{"col_name":"Part_Status", "data_type":"string","comment":"[ID:{3108}]"}
    ,{"col_name":"Price","data_type":"decimal(16,6)","comment":"[ID:{33720}]"}
    ,{"col_name":"Active", "data_type":"smallint","comment":"[ID:{14452}]"}
    ,{"col_name":"Expiration_Date", "data_type":"timestamp","comment":"[ID:{27721}]"}
    ,{"col_name":"Unit", "data_type":"string","comment":"[ID:{89167}]"}
    ,{"col_name":"Add_Date", "data_type":"timestamp","comment":"[ID:{48923}]"}
    ,{"col_name":"Update_Date", "data_type":"timestamp","comment":"[ID:{7139}]"}
    ,{"col_name":"Cost","data_type":"decimal(16,6)","comment":"[ID:{73019}]"}
    ,{"col_name":"Margin","data_type":"decimal(16,6)","comment":"[ID:{52122}]"}
    ,{"col_name":"Part_Cost","data_type":"decimal(16,6)","comment":"[ID:{52123}]"}
    ,{"col_name":"Price_In_Effect","data_type":"boolean","comment":"[ID:{33714}]"}
    ,{"col_name":"Price_In_Effect_Basis","data_type":"decimal(16,6)","comment":"[ID:{33705}]"}
    ,{"col_name":"AuditPipelineTriggerTime", "data_type":"date","comment":"[ID:{30231}]"}
	,{"col_name":"AuditPipelineRunId", "data_type":"string","comment":"[ID:{62840}]"}
	,{"col_name":"AuditFilePath", "data_type":"string","comment":"[ID:{62216}]"}
	,{"col_name":"Audit_Source_ModifiedDateTime", "data_type":"timestamp","comment":""}
	,{"col_name":"RevisionHash", "data_type":"binary","comment":""}
	,{"col_name":"Audit_SourceFileOrFolder", "data_type":"string","comment":""}
	,{"col_name":"Audit_CreatedDateTime", "data_type":"timestamp","comment":""}
	,{"col_name":"Audit_ModifiedDateTime", "data_type":"timestamp","comment":""}
]
partitions=[]
path=destination_filepath+"price"

ProcessTable("LH_BI_SILVER_TIER1","price",comparisonArray,partitions,path)

StatementMeta(, e00e84cf-a8f6-49bf-a364-9470ed7caf20, 51, Finished, Available, Finished, False)

# Process_workcentre

In [50]:
comparisonArray=[
{"col_name":"Plexus_Customer_No","data_type":"int","comment":"[ID:{92319}]"}
,{"col_name":"Workcenter_Key","data_type":"int","comment":"[ID:{92320}]"}
,{"col_name":"Workcenter_Code","data_type":"string","comment":"[ID:{92321}]"}
,{"col_name":"Name","data_type":"string","comment":"[ID:{94242}]"}
,{"col_name":"Workcenter_Type","data_type":"string","comment":"[ID:{33711}]"}
,{"col_name":"Part_Key","data_type":"int","comment":"[ID:{19267}]"}
,{"col_name":"Part_Operation_Key","data_type":"int","comment":"[ID:{33710}]"}
,{"col_name":"Building_Key","data_type":"int","comment":"[ID:{33712}]"}
,{"col_name":"Shift_Cycle_Key","data_type":"int","comment":"[ID:{33713}]"}
,{"col_name":"Variable_Cost","data_type":"decimal(16,6)","comment":"[ID:{73019}]"}
,{"col_name":"Overhead_Cost","data_type":"decimal(16,6)","comment":"[ID:{52122}]"}
,{"col_name":"Cost_Unit","data_type":"string","comment":"[ID:{52123}]"}
,{"col_name":"Account_No","data_type":"string","comment":"[ID:{33714}]"}
,{"col_name":"Workcenter_Group","data_type":"string","comment":"[ID:{33119}}]"}
,{"col_name":"Active","data_type":"smallint","comment":"[ID:{54553}]"}
,{"col_name":"Efficiency","data_type":"decimal(16,6)","comment":"[ID:{40356}]"}
,{"col_name":"Department_No","data_type":"int","comment":"[ID:{82508}]"}
,{"col_name":"Direct_Labor_Cost","data_type":"decimal(16,6)","comment":"[ID:{23382}]"}

,{"col_name":"AuditPipelineTriggerTime", "data_type":"date","comment":"[ID:{59935}]"}
,{"col_name":"AuditPipelineRunId", "data_type":"string","comment":"[ID:{24411}]"}
,{"col_name":"AuditFilePath", "data_type":"string","comment":"[ID:{96427}]"}
,{"col_name":"Audit_Source_ModifiedDateTime", "data_type":"timestamp","comment":""}
,{"col_name":"RevisionHash", "data_type":"binary","comment":""}
,{"col_name":"Audit_SourceFileOrFolder", "data_type":"string","comment":""}
,{"col_name":"Audit_CreatedDateTime", "data_type":"timestamp","comment":""}
,{"col_name":"Audit_ModifiedDateTime", "data_type":"timestamp","comment":""}
]
partitions=[]
path=destination_filepath+"workcenter"

ProcessTable("LH_BI_SILVER_TIER1","workcenter",comparisonArray,partitions,path)

StatementMeta(, e00e84cf-a8f6-49bf-a364-9470ed7caf20, 52, Finished, Available, Finished, False)

#### Process table Audit_RuntimeTableLog.

In [51]:
comparisonArray=[
	{"col_name":"GroupingID", "data_type":"string","comment":""}
    ,{"col_name":"StartDateTime", "data_type":"timestamp","comment":""}
	,{"col_name":"DatabaseName", "data_type":"string","comment":""}
    ,{"col_name":"TableName", "data_type":"string","comment":""}
    ,{"col_name":"TableStartTime", "data_type":"timestamp","comment":""}
    ,{"col_name":"TableEndTime", "data_type":"timestamp","comment":""}
    ,{"col_name":"TableDurationSeconds", "data_type":"bigint","comment":""}
    ,{"col_name":"RowsAffected", "data_type":"bigint","comment":""}
    ,{"col_name":"RowsInserted", "data_type":"bigint","comment":""}
    ,{"col_name":"RowsUpdated", "data_type":"bigint","comment":""}
    ,{"col_name":"RowsDeleted", "data_type":"bigint","comment":""}
    ,{"col_name":"RowsBefore", "data_type":"bigint","comment":""}
    ,{"col_name":"RowsAfter", "data_type":"bigint","comment":""}
    ,{"col_name":"Audit_Status", "data_type":"string","comment":""}
    ,{"col_name":"Audit_CreatedDateTime", "data_type":"timestamp","comment":""}
	,{"col_name":"Audit_ModifiedDateTime", "data_type":"timestamp","comment":""}
]
partitions=["StartDateTime","DatabaseName","TableName"]

path=destination_filepath+"Audit_RuntimeTableLog"

ProcessTable("LH_BI_SILVER_TIER1","Audit_RuntimeTableLog",comparisonArray,partitions,path)

StatementMeta(, e00e84cf-a8f6-49bf-a364-9470ed7caf20, 53, Finished, Available, Finished, False)

#### Process table Audit_HighWaterMark.

In [52]:
comparisonArray=[
	{"col_name":"Type", "data_type":"string","comment":""}
    ,{"col_name":"DatabaseName", "data_type":"string","comment":""}
	,{"col_name":"TableName", "data_type":"string","comment":""}
    ,{"col_name":"Value", "data_type":"string","comment":""}
    ,{"col_name":"Audit_CreatedDateTime", "data_type":"timestamp","comment":""}
	,{"col_name":"Audit_ModifiedDateTime", "data_type":"timestamp","comment":""}
]
partitions=["DatabaseName","TableName"]

path=destination_filepath+"Audit_HighWaterMark"

ProcessTable("LH_BI_SILVER_TIER1","Audit_HighWaterMark",comparisonArray,partitions,path)

StatementMeta(, e00e84cf-a8f6-49bf-a364-9470ed7caf20, 54, Finished, Available, Finished, False)

#### End of Notebook